# 1

In [1]:
# 1 ============================================================
# ✅ DATA FIX + MODELING (Block-selection, not row filtering)
# ------------------------------------------------------------
# Fixes your situation:
# - Non-depressed rows have missing predictors because you're reading the wrong column-block
# - We select, for each feature concept, the most-populated column among duplicates
# - Then build X,y and run Group-safe CV
# ============================================================

import warnings, re, hashlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from collections import Counter

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.calibration import CalibratedClassifierCV

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    precision_score, recall_score, f1_score, fbeta_score, confusion_matrix, make_scorer
)
from sklearn.model_selection import RandomizedSearchCV

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# Optional XGBoost
use_xgboost = False
try:
    from xgboost import XGBClassifier
    use_xgboost = True
except Exception:
    use_xgboost = False

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# -------------------------------
# 0) CONFIG
# -------------------------------
COHORT = "antepartum"   # "antepartum" or "postpartum"
ANTE_PATH = "/kaggle/input/datasets/maishaidaralkhateeb/ppd-depression/Antepartum_Merged_from_Postpartum.xlsx"
POST_PATH = "/kaggle/input/ppd-depression/Postpartum Depression.xlsx"

EPDS_CUTOFFS = {"antepartum": 12, "postpartum": 9}
cutoff = EPDS_CUTOFFS[COHORT]

LEAK_PATTERNS = ["depress", "depression", "without", "outcome", "label"]  # keep EPDS for y only

N_SPLITS = 5
N_REPEATS = 3
INNER_VAL_FRACTION = 0.20
BETA = 2.0

# -------------------------------
# 1) Load + clean headers
# -------------------------------
def load_and_clean_excel(path: str) -> pd.DataFrame:
    df = pd.read_excel(path, header=0)
    df = df.dropna(axis=1, how="all")
    df.columns = (
        df.columns.astype(str)
        .str.strip().str.lower()
        .str.replace(" ", "_")
        .str.replace(r"[^0-9a-zA-Z_]", "", regex=True)
    )
    df = df.loc[:, ~df.columns.duplicated()].copy()
    df = df.drop_duplicates()
    return df

df = load_and_clean_excel(ANTE_PATH if COHORT == "antepartum" else POST_PATH)
print("Loaded:", COHORT, "shape:", df.shape)
print("Sample columns:", df.columns.tolist()[:15])

# -------------------------------
# 2) Detect EPDS column for label
# -------------------------------
def find_best_match(cols, keywords):
    for kw in keywords:
        hits = [c for c in cols if kw in c]
        if hits:
            return hits[0]
    return None

if COHORT == "antepartum":
    epds_col = find_best_match(df.columns, ["epds_antepartum", "epdsantepartum", "epds"])
else:
    epds_col = find_best_match(df.columns, ["epds_postpartum", "epdspostpartum", "epds"])

if epds_col is None:
    raise ValueError("Could not find EPDS column.")

df[epds_col] = pd.to_numeric(df[epds_col], errors="coerce")
y = (df[epds_col] >= cutoff).astype(int)

print("Detected EPDS:", epds_col, "| cutoff:", cutoff)
print("y distribution:", Counter(y))

# -------------------------------
# 3) Select the most-populated column per concept
# -------------------------------
def best_populated(df, candidates):
    """Pick the candidate column with the highest non-missing count after numeric coercion."""
    best = None
    best_n = -1
    for c in candidates:
        if c not in df.columns:
            continue
        s = pd.to_numeric(df[c], errors="coerce")
        n = int(s.notna().sum())
        if n > best_n:
            best_n = n
            best = c
    return best, best_n

# Feature concepts and their possible column names (you can extend this list)
FEATURE_MAP = {
    "age": ["age", "age1"],
    "anxious_attachment": ["anxious_attachment", "anxious_a", "anxious_a1"],
    "avoidant_attachment": ["avoidant_attachment", "avoidant_a", "avoidant_a1"],
    "reframing": ["reframing", "positive_reframing", "reframing1"],
    "selfdistraction": ["selfdistraction", "selfdistraction1"],
    "venting": ["venting", "venting1"],
    "use_of_informational_support": ["use_of_informational_support", "use_of_informational", "use_of_informational1"],
    "active_c": ["active_c", "active", "active_c1", "active1"],
    "denial": ["denial", "denial1"],
    "religion": ["religion", "religion1"],
    "humour": ["humour", "humour1"],
    "disengagement": ["disengagement", "disengagement1"],
    "use_of_emotional_s": ["use_of_emotional_s", "emotional_s", "emotional_s1"],
    "substance": ["substance", "substance1"],
    "acceptance": ["acceptance", "acceptance1"],
    "planning": ["planning", "planning1"],
    "selfblame": ["selfblame", "self_blame", "selfblame1"],
}

selected = {}
for feat, cands in FEATURE_MAP.items():
    col, n = best_populated(df, cands)
    if col is not None and n > 0:
        selected[feat] = col

print("\nSelected columns (most populated per concept):")
for k, v in selected.items():
    nn = int(pd.to_numeric(df[v], errors="coerce").notna().sum())
    print(f"  {k:28s} -> {v:35s}  non-missing={nn}")

# -------------------------------
# 4) Build X using selected columns only
# -------------------------------
X = pd.DataFrame(index=df.index)
for feat, col in selected.items():
    X[feat] = pd.to_numeric(df[col], errors="coerce")

# Drop leakage proxy columns if any accidentally got in (shouldn't)
drop_like = [c for c in X.columns if any(p in c for p in LEAK_PATTERNS)]
if drop_like:
    X = X.drop(columns=drop_like, errors="ignore")

# Keep rows with at least SOME observed values
row_keep = X.notna().sum(axis=1) >= 3
X = X.loc[row_keep].copy()
y = y.loc[row_keep].copy()

print("\nFinal X shape:", X.shape)
print("Final y distribution:", Counter(y))

if y.nunique() < 2:
    raise ValueError("Still only one class after selecting populated columns. The file is structurally broken.")

# -------------------------------
# 5) Re-check missingness by class (should NOT be 1.0 gap anymore)
# -------------------------------
overall_miss = X.isna().mean().sort_values(ascending=False)
miss_by_class = X.isna().groupby(y).mean()
miss_by_class.index = ["Non-depressed(0)", "Depressed(1)"]
gap = (miss_by_class.loc["Depressed(1)"] - miss_by_class.loc["Non-depressed(0)"]).abs().sort_values(ascending=False)

print("\nMissingness gap top 10 (should be small-ish now):")
print(gap.head(10))

# -------------------------------
# 6) Groups by feature-row hash
# -------------------------------
X_for_hash = X.fillna("__MISSING__").astype(str)
groups = X_for_hash.apply(lambda r: hashlib.md5("||".join(r.values.tolist()).encode("utf-8")).hexdigest(), axis=1)
groups = pd.Series(groups, index=X.index)

print("\n[GROUP CHECK] Unique groups:", groups.nunique(), "out of", len(groups))
print("[GROUP CHECK] Largest group size:", int(groups.value_counts().max()))

# -------------------------------
# 7) Splitters
# -------------------------------
try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAS_SGKF = True
except Exception:
    HAS_SGKF = False
    from sklearn.model_selection import GroupKFold

def outer_splitter(seed: int):
    if HAS_SGKF:
        cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
        return cv.split(X, y, groups=groups)
    else:
        gkf = GroupKFold(n_splits=N_SPLITS)
        return gkf.split(X, y, groups=groups)

def group_safe_inner_split(train_idx, val_frac, seed):
    rng = np.random.default_rng(seed)
    train_groups = groups.iloc[train_idx]
    uniq_g = train_groups.unique()
    rng.shuffle(uniq_g)
    n_val_groups = max(1, int(len(uniq_g) * val_frac))
    val_g = set(uniq_g[:n_val_groups])
    inner_val_mask = train_groups.isin(val_g).values
    inner_val_idx = train_idx[inner_val_mask]
    inner_train_idx = train_idx[~inner_val_mask]
    return inner_train_idx, inner_val_idx

# -------------------------------
# 8) Models + pipelines
# -------------------------------
def make_base_estimator(model_name, mode, spw):
    if model_name == "LR":
        cw = "balanced" if mode == "weighted" else None
        return LogisticRegression(max_iter=5000, class_weight=cw, random_state=RANDOM_STATE)
    if model_name == "SVM":
        cw = "balanced" if mode == "weighted" else None
        return SVC(kernel="rbf", probability=True, class_weight=cw, random_state=RANDOM_STATE)
    if model_name == "RF":
        cw = "balanced" if mode == "weighted" else None
        return RandomForestClassifier(n_estimators=400, class_weight=cw, random_state=RANDOM_STATE, n_jobs=-1)
    if model_name == "XGB":
        if not use_xgboost:
            return HistGradientBoostingClassifier(random_state=RANDOM_STATE)
        spw2 = spw if mode == "weighted" else 1.0
        return XGBClassifier(
            n_estimators=500, max_depth=3, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
            scale_pos_weight=spw2, eval_metric="logloss",
            random_state=RANDOM_STATE, n_jobs=-1
        )
    return HistGradientBoostingClassifier(random_state=RANDOM_STATE)

def build_pipeline(model_name, mode, spw):
    imputer = ("imputer", SimpleImputer(strategy="median"))
    scaler = ("scaler", StandardScaler())
    estimator = ("model", make_base_estimator(model_name, mode, spw))

    if mode == "resampled":
        smote = ("smote", SMOTE(sampling_strategy=0.8, random_state=RANDOM_STATE))
        under = ("under", RandomUnderSampler(sampling_strategy=1.0, random_state=RANDOM_STATE))
        return ImbPipeline([imputer, scaler, smote, under, estimator])
    return SkPipeline([imputer, scaler, estimator])

def get_param_distributions(model_name):
    if model_name == "LR":
        return {"model__C": np.logspace(-3, 3, 25)}
    if model_name == "SVM":
        return {"model__C": np.logspace(-2, 3, 20), "model__gamma": ["scale", "auto"]}
    if model_name == "RF":
        return {"model__max_depth": [None, 4, 7, 12], "model__min_samples_leaf": [1, 2, 4]}
    if model_name == "XGB":
        return {"model__max_depth": [2, 3, 4], "model__learning_rate": [0.03, 0.05, 0.1]}
    return {}

def choose_threshold_on_val(y_val, p_val, beta=BETA):
    thresholds = np.linspace(0.05, 0.95, 91)
    best_t, best_score = 0.5, -1.0
    for t in thresholds:
        y_hat = (p_val >= t).astype(int)
        score = fbeta_score(y_val, y_hat, beta=beta, zero_division=0)
        if score > best_score:
            best_score = score
            best_t = t
    return best_t, best_score

# -------------------------------
# 9) Label-shuffle sanity test (quick)
# -------------------------------
def quick_groupcv_auc(seed=42, shuffle_labels=False):
    rng = np.random.default_rng(seed)
    y_tested = y.copy()
    if shuffle_labels:
        y_tested = pd.Series(rng.permutation(y_tested.values), index=y_tested.index)

    aucs, aps = [], []
    for tr_idx, te_idx in outer_splitter(seed):
        X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
        y_tr, y_te = y_tested.iloc[tr_idx], y_tested.iloc[te_idx]
        n_pos = int((y_tr == 1).sum())
        n_neg = int((y_tr == 0).sum())
        spw = (n_neg / max(n_pos, 1))

        pipe = build_pipeline("SVM", "weighted", spw)
        pipe.fit(X_tr, y_tr)
        p = pipe.predict_proba(X_te)[:, 1]
        aucs.append(roc_auc_score(y_te, p))
        aps.append(average_precision_score(y_te, p))
    return float(np.mean(aucs)), float(np.mean(aps))

print("\n[SHUFFLE SANITY]")
ra, rp = quick_groupcv_auc(seed=42, shuffle_labels=False)
sa, sp = quick_groupcv_auc(seed=42, shuffle_labels=True)
print(f"Real     -> ROC-AUC={ra:.3f}, PR-AUC={rp:.3f}")
print(f"Shuffled -> ROC-AUC={sa:.3f}, PR-AUC={sp:.3f}")

# -------------------------------
# 10) Full evaluation
# -------------------------------
model_specs = ["LR", "SVM", "RF", "XGB"]
modes = ["weighted", "resampled"]

f2_scorer = make_scorer(fbeta_score, beta=BETA, zero_division=0)

rows = []

for rep in range(N_REPEATS):
    rep_seed = RANDOM_STATE + rep
    print(f"\n===== REPEAT {rep+1}/{N_REPEATS} | seed={rep_seed} =====")

    for model_name in model_specs:
        for mode in modes:
            print(f"\n=== {COHORT.upper()} | MODEL={model_name} | MODE={mode} ===")

            for fold, (train_idx, test_idx) in enumerate(outer_splitter(rep_seed), start=1):
                X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
                y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

                n_pos = int((y_train == 1).sum())
                n_neg = int((y_train == 0).sum())
                spw = (n_neg / max(n_pos, 1))

                inner_train_idx, inner_val_idx = group_safe_inner_split(train_idx, INNER_VAL_FRACTION, seed=rep_seed + 1000 + fold)
                X_tr, y_tr = X.iloc[inner_train_idx], y.iloc[inner_train_idx]
                X_val, y_val = X.iloc[inner_val_idx], y.iloc[inner_val_idx]

                pipe = build_pipeline(model_name, mode, spw)
                param_dist = get_param_distributions(model_name)

                search = RandomizedSearchCV(
                    estimator=pipe,
                    param_distributions=param_dist,
                    n_iter=15,
                    scoring=f2_scorer,
                    cv=3,
                    random_state=rep_seed,
                    n_jobs=-1,
                    refit=True
                )
                search.fit(X_tr, y_tr)
                best_pipe = search.best_estimator_

                cal = CalibratedClassifierCV(best_pipe, method="sigmoid", cv=3)
                cal.fit(X_tr, y_tr)

                p_val = cal.predict_proba(X_val)[:, 1]
                thr, _ = choose_threshold_on_val(y_val.values, p_val, beta=BETA)

                p_test = cal.predict_proba(X_test)[:, 1]
                y_hat = (p_test >= thr).astype(int)

                roc_auc = roc_auc_score(y_test, p_test)
                pr_auc = average_precision_score(y_test, p_test)
                brier = brier_score_loss(y_test, p_test)

                prec = precision_score(y_test, y_hat, zero_division=0)
                rec = recall_score(y_test, y_hat, zero_division=0)
                f1 = f1_score(y_test, y_hat, zero_division=0)
                f2 = fbeta_score(y_test, y_hat, beta=BETA, zero_division=0)

                tn, fp, fn, tp = confusion_matrix(y_test, y_hat).ravel()
                spec = tn / (tn + fp) if (tn + fp) else 0.0

                rows.append({
                    "cohort": COHORT, "repeat": rep+1, "fold": fold,
                    "model": model_name, "mode": mode,
                    "roc_auc": roc_auc, "pr_auc": pr_auc, "brier": brier,
                    "precision": prec, "recall": rec, "specificity": spec,
                    "f1": f1, "f2": f2, "thr": thr
                })

                print(f"Rep {rep+1} Fold {fold:02d} | thr={thr:.2f} | PR-AUC={pr_auc:.3f} | ROC-AUC={roc_auc:.3f} | "
                      f"Brier={brier:.3f} | R={rec:.3f} P={prec:.3f}")

results_df = pd.DataFrame(rows)
summary = (results_df.groupby(["cohort","model","mode"])
           .agg(roc_auc_mean=("roc_auc","mean"), pr_auc_mean=("pr_auc","mean"),
                brier_mean=("brier","mean"), recall_mean=("recall","mean"),
                precision_mean=("precision","mean"), f2_mean=("f2","mean"),
                thr_mean=("thr","mean"), folds=("thr","count"))
           .reset_index()
           .sort_values("recall_mean", ascending=False))

print("\n=== SUMMARY ===")
print(summary.to_string(index=False))

Loaded: antepartum shape: (1430, 22)
Sample columns: ['match_key', 'antepartum__depression', 'age', 'epds_antepartum', 'anxious_a', 'avoidant_a', 'reframing', 'selfdistraction', 'venting', 'use_of_informational', 'active', 'denial', 'religion', 'humour', 'disengagement']
Detected EPDS: epds_antepartum | cutoff: 12
y distribution: Counter({0: 1196, 1: 234})

Selected columns (most populated per concept):
  age                          -> age                                  non-missing=1332
  anxious_attachment           -> anxious_a                            non-missing=1430
  avoidant_attachment          -> avoidant_a                           non-missing=1430
  reframing                    -> reframing                            non-missing=1430
  selfdistraction              -> selfdistraction                      non-missing=1332
  venting                      -> venting                              non-missing=1430
  use_of_informational_support -> use_of_informational           

# [](http://)2

In [2]:
# ============================================================
# ✅ DATA FIX + MODELING (Block-selection) + ✅ PROXY DETECTOR
# ------------------------------------------------------------
# What this script does:
# ✅ Label from EPDS cutoff (no OUTCOME)
# ✅ Block-selection: for each "concept" pick the most-populated column among duplicates
# ✅ Group-safe CV (StratifiedGroupKFold if available)
# ✅ Inner validation split is group-safe (NO test-set threshold tuning)
# ✅ Randomized hyperparameter search
# ✅ Calibration (sigmoid) + Brier
# ✅ ROC-AUC + PR-AUC
# ✅ Label-shuffle sanity test
# ✅ PROXY DETECTOR: single-feature AUC scan (find what makes trees perfect)
# ============================================================

import warnings, re, hashlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from collections import Counter

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.calibration import CalibratedClassifierCV

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    precision_score, recall_score, f1_score, fbeta_score, confusion_matrix, make_scorer
)
from sklearn.model_selection import RandomizedSearchCV

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# Optional XGBoost
use_xgboost = False
try:
    from xgboost import XGBClassifier
    use_xgboost = True
except Exception:
    use_xgboost = False

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# -------------------------------
# 0) CONFIG
# -------------------------------
COHORT = "antepartum"   # "antepartum" or "postpartum"
ANTE_PATH = "/kaggle/input/datasets/maishaidaralkhateeb/ppd-depression/Antepartum_Merged_from_Postpartum.xlsx"
POST_PATH = "/kaggle/input/ppd-depression/Postpartum Depression.xlsx"

EPDS_CUTOFFS = {"antepartum": 12, "postpartum": 9}
cutoff = EPDS_CUTOFFS[COHORT]

# We keep EPDS column only for y creation, then drop it from X
LEAK_PATTERNS = ["depress", "depression", "without", "outcome", "label"]

# CV settings
N_SPLITS = 5
N_REPEATS = 3
INNER_VAL_FRACTION = 0.20
BETA = 2.0  # F2 emphasis

# -------------------------------
# 1) Load + clean headers
# -------------------------------
def load_and_clean_excel(path: str) -> pd.DataFrame:
    df = pd.read_excel(path, header=0)
    df = df.dropna(axis=1, how="all")
    df.columns = (
        df.columns.astype(str)
        .str.strip().str.lower()
        .str.replace(" ", "_")
        .str.replace(r"[^0-9a-zA-Z_]", "", regex=True)
    )
    df = df.loc[:, ~df.columns.duplicated()].copy()
    df = df.drop_duplicates()
    return df

df = load_and_clean_excel(ANTE_PATH if COHORT == "antepartum" else POST_PATH)
print("Loaded:", COHORT, "shape:", df.shape)
print("Sample columns:", df.columns.tolist()[:15])

# -------------------------------
# 2) Detect EPDS column for label
# -------------------------------
def find_best_match(cols, keywords):
    for kw in keywords:
        hits = [c for c in cols if kw in c]
        if hits:
            return hits[0]
    return None

if COHORT == "antepartum":
    epds_col = find_best_match(df.columns, ["epds_antepartum", "epdsantepartum", "epds"])
else:
    epds_col = find_best_match(df.columns, ["epds_postpartum", "epdspostpartum", "epds"])

if epds_col is None:
    raise ValueError("Could not find EPDS column.")

df[epds_col] = pd.to_numeric(df[epds_col], errors="coerce")
y = (df[epds_col] >= cutoff).astype(int)

print("Detected EPDS:", epds_col, "| cutoff:", cutoff)
print("y distribution:", Counter(y))

# -------------------------------
# 3) Select the most-populated column per concept (block selection)
# -------------------------------
def best_populated(df, candidates):
    """Pick candidate column with highest non-missing count after numeric coercion."""
    best = None
    best_n = -1
    for c in candidates:
        if c not in df.columns:
            continue
        s = pd.to_numeric(df[c], errors="coerce")
        n = int(s.notna().sum())
        if n > best_n:
            best_n = n
            best = c
    return best, best_n

FEATURE_MAP = {
    "age": ["age", "age1"],
    "anxious_attachment": ["anxious_attachment", "anxious_a", "anxious_a1"],
    "avoidant_attachment": ["avoidant_attachment", "avoidant_a", "avoidant_a1"],
    "reframing": ["reframing", "positive_reframing", "reframing1"],
    "selfdistraction": ["selfdistraction", "selfdistraction1"],
    "venting": ["venting", "venting1"],
    "use_of_informational_support": ["use_of_informational_support", "use_of_informational", "use_of_informational1"],
    "active_c": ["active_c", "active", "active_c1", "active1"],
    "denial": ["denial", "denial1"],
    "religion": ["religion", "religion1"],
    "humour": ["humour", "humour1"],
    "disengagement": ["disengagement", "disengagement1"],
    "use_of_emotional_s": ["use_of_emotional_s", "emotional_s", "emotional_s1"],
    "substance": ["substance", "substance1"],
    "acceptance": ["acceptance", "acceptance1"],
    "planning": ["planning", "planning1"],
    "selfblame": ["selfblame", "self_blame", "selfblame1"],
}

selected = {}
for feat, cands in FEATURE_MAP.items():
    col, n = best_populated(df, cands)
    if col is not None and n > 0:
        selected[feat] = col

print("\nSelected columns (most populated per concept):")
for feat, col in selected.items():
    nn = int(pd.to_numeric(df[col], errors="coerce").notna().sum())
    print(f"  {feat:28s} -> {col:35s}  non-missing={nn}")

# -------------------------------
# 4) Build X using selected columns only
# -------------------------------
X = pd.DataFrame(index=df.index)
for feat, col in selected.items():
    X[feat] = pd.to_numeric(df[col], errors="coerce")

# Remove any accidentally-leaky feature names (should be none since we control the map)
drop_like = [c for c in X.columns if any(p in c for p in LEAK_PATTERNS)]
if drop_like:
    X = X.drop(columns=drop_like, errors="ignore")

# Keep rows with at least SOME observed values
row_keep = X.notna().sum(axis=1) >= 3
X = X.loc[row_keep].copy()
y = y.loc[row_keep].copy()

print("\nFinal X shape:", X.shape)
print("Final y distribution:", Counter(y))

if y.nunique() < 2:
    raise ValueError("Only one class remains after block selection. The Excel sheet is structurally broken.")

# -------------------------------
# 5) Missingness gap check
# -------------------------------
overall_miss = X.isna().mean().sort_values(ascending=False)
miss_by_class = X.isna().groupby(y).mean()
miss_by_class.index = ["Non-depressed(0)", "Depressed(1)"]
gap = (miss_by_class.loc["Depressed(1)"] - miss_by_class.loc["Non-depressed(0)"]).abs().sort_values(ascending=False)

print("\nMissingness gap top 10 (should not be 1.0 now):")
print(gap.head(10))

# -------------------------------
# 6) Groups by feature-row hash
# -------------------------------
X_for_hash = X.fillna("__MISSING__").astype(str)
groups = X_for_hash.apply(lambda r: hashlib.md5("||".join(r.values.tolist()).encode("utf-8")).hexdigest(), axis=1)
groups = pd.Series(groups, index=X.index)

print("\n[GROUP CHECK] Unique groups:", groups.nunique(), "out of", len(groups))
print("[GROUP CHECK] Largest group size:", int(groups.value_counts().max()))

# -------------------------------
# 7) Splitters (StratifiedGroupKFold if available)
# -------------------------------
try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAS_SGKF = True
except Exception:
    HAS_SGKF = False
    from sklearn.model_selection import GroupKFold

def outer_splitter(seed: int):
    if HAS_SGKF:
        cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
        return cv.split(X, y, groups=groups)
    else:
        gkf = GroupKFold(n_splits=N_SPLITS)
        return gkf.split(X, y, groups=groups)

def group_safe_inner_split(train_idx, val_frac, seed):
    rng = np.random.default_rng(seed)
    train_groups = groups.iloc[train_idx]
    uniq_g = train_groups.unique()
    rng.shuffle(uniq_g)
    n_val_groups = max(1, int(len(uniq_g) * val_frac))
    val_g = set(uniq_g[:n_val_groups])
    inner_val_mask = train_groups.isin(val_g).values
    inner_val_idx = train_idx[inner_val_mask]
    inner_train_idx = train_idx[~inner_val_mask]
    return inner_train_idx, inner_val_idx

# -------------------------------
# 8) Models + pipelines
# -------------------------------
def make_base_estimator(model_name, mode, spw):
    if model_name == "LR":
        cw = "balanced" if mode == "weighted" else None
        return LogisticRegression(max_iter=5000, class_weight=cw, random_state=RANDOM_STATE)
    if model_name == "SVM":
        cw = "balanced" if mode == "weighted" else None
        return SVC(kernel="rbf", probability=True, class_weight=cw, random_state=RANDOM_STATE)
    if model_name == "RF":
        cw = "balanced" if mode == "weighted" else None
        return RandomForestClassifier(n_estimators=400, class_weight=cw, random_state=RANDOM_STATE, n_jobs=-1)
    if model_name == "XGB":
        if not use_xgboost:
            return HistGradientBoostingClassifier(random_state=RANDOM_STATE)
        spw2 = spw if mode == "weighted" else 1.0
        return XGBClassifier(
            n_estimators=500, max_depth=3, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
            scale_pos_weight=spw2, eval_metric="logloss",
            random_state=RANDOM_STATE, n_jobs=-1
        )
    return HistGradientBoostingClassifier(random_state=RANDOM_STATE)

def build_pipeline(model_name, mode, spw):
    imputer = ("imputer", SimpleImputer(strategy="median"))
    scaler = ("scaler", StandardScaler())
    estimator = ("model", make_base_estimator(model_name, mode, spw))

    if mode == "resampled":
        smote = ("smote", SMOTE(sampling_strategy=0.8, random_state=RANDOM_STATE))
        under = ("under", RandomUnderSampler(sampling_strategy=1.0, random_state=RANDOM_STATE))
        return ImbPipeline([imputer, scaler, smote, under, estimator])
    return SkPipeline([imputer, scaler, estimator])

def get_param_distributions(model_name):
    if model_name == "LR":
        return {"model__C": np.logspace(-3, 3, 25)}
    if model_name == "SVM":
        return {"model__C": np.logspace(-2, 3, 20), "model__gamma": ["scale", "auto"]}
    if model_name == "RF":
        return {"model__max_depth": [None, 4, 7, 12], "model__min_samples_leaf": [1, 2, 4]}
    if model_name == "XGB":
        return {"model__max_depth": [2, 3, 4], "model__learning_rate": [0.03, 0.05, 0.1]}
    return {}

def choose_threshold_on_val(y_val, p_val, beta=BETA):
    thresholds = np.linspace(0.05, 0.95, 91)
    best_t, best_score = 0.5, -1.0
    for t in thresholds:
        y_hat = (p_val >= t).astype(int)
        score = fbeta_score(y_val, y_hat, beta=beta, zero_division=0)
        if score > best_score:
            best_score = score
            best_t = t
    return best_t, best_score

# -------------------------------
# 9) Label-shuffle sanity test (quick)
# -------------------------------
def quick_groupcv_auc(seed=42, shuffle_labels=False):
    rng = np.random.default_rng(seed)
    y_tested = y.copy()
    if shuffle_labels:
        y_tested = pd.Series(rng.permutation(y_tested.values), index=y_tested.index)

    aucs, aps = [], []
    for tr_idx, te_idx in outer_splitter(seed):
        X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
        y_tr, y_te = y_tested.iloc[tr_idx], y_tested.iloc[te_idx]

        n_pos = int((y_tr == 1).sum())
        n_neg = int((y_tr == 0).sum())
        spw = (n_neg / max(n_pos, 1))

        pipe = build_pipeline("SVM", "weighted", spw)
        pipe.fit(X_tr, y_tr)
        p = pipe.predict_proba(X_te)[:, 1]
        aucs.append(roc_auc_score(y_te, p))
        aps.append(average_precision_score(y_te, p))
    return float(np.mean(aucs)), float(np.mean(aps))

print("\n[SHUFFLE SANITY]")
ra, rp = quick_groupcv_auc(seed=42, shuffle_labels=False)
sa, sp = quick_groupcv_auc(seed=42, shuffle_labels=True)
print(f"Real     -> ROC-AUC={ra:.3f}, PR-AUC={rp:.3f}")
print(f"Shuffled -> ROC-AUC={sa:.3f}, PR-AUC={sp:.3f}")

# -------------------------------
# 10) Full evaluation (Group-safe)
# -------------------------------
model_specs = ["LR", "SVM", "RF", "XGB"]
modes = ["weighted", "resampled"]

f2_scorer = make_scorer(fbeta_score, beta=BETA, zero_division=0)
rows = []

for rep in range(N_REPEATS):
    rep_seed = RANDOM_STATE + rep
    print(f"\n===== REPEAT {rep+1}/{N_REPEATS} | seed={rep_seed} =====")

    for model_name in model_specs:
        for mode in modes:
            print(f"\n=== {COHORT.upper()} | MODEL={model_name} | MODE={mode} ===")

            for fold, (train_idx, test_idx) in enumerate(outer_splitter(rep_seed), start=1):
                X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
                y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

                n_pos = int((y_train == 1).sum())
                n_neg = int((y_train == 0).sum())
                spw = (n_neg / max(n_pos, 1))

                inner_train_idx, inner_val_idx = group_safe_inner_split(
                    train_idx, INNER_VAL_FRACTION, seed=rep_seed + 1000 + fold
                )
                X_tr, y_tr = X.iloc[inner_train_idx], y.iloc[inner_train_idx]
                X_val, y_val = X.iloc[inner_val_idx], y.iloc[inner_val_idx]

                pipe = build_pipeline(model_name, mode, spw)
                param_dist = get_param_distributions(model_name)

                search = RandomizedSearchCV(
                    estimator=pipe,
                    param_distributions=param_dist,
                    n_iter=15,
                    scoring=f2_scorer,
                    cv=3,
                    random_state=rep_seed,
                    n_jobs=-1,
                    refit=True
                )
                search.fit(X_tr, y_tr)
                best_pipe = search.best_estimator_

                cal = CalibratedClassifierCV(best_pipe, method="sigmoid", cv=3)
                cal.fit(X_tr, y_tr)

                p_val = cal.predict_proba(X_val)[:, 1]
                thr, _ = choose_threshold_on_val(y_val.values, p_val, beta=BETA)

                p_test = cal.predict_proba(X_test)[:, 1]
                y_hat = (p_test >= thr).astype(int)

                roc_auc = roc_auc_score(y_test, p_test)
                pr_auc = average_precision_score(y_test, p_test)
                brier = brier_score_loss(y_test, p_test)

                prec = precision_score(y_test, y_hat, zero_division=0)
                rec = recall_score(y_test, y_hat, zero_division=0)
                f1 = f1_score(y_test, y_hat, zero_division=0)
                f2 = fbeta_score(y_test, y_hat, beta=BETA, zero_division=0)

                tn, fp, fn, tp = confusion_matrix(y_test, y_hat).ravel()
                spec = tn / (tn + fp) if (tn + fp) else 0.0

                rows.append({
                    "cohort": COHORT, "repeat": rep+1, "fold": fold,
                    "model": model_name, "mode": mode,
                    "roc_auc": roc_auc, "pr_auc": pr_auc, "brier": brier,
                    "precision": prec, "recall": rec, "specificity": spec,
                    "f1": f1, "f2": f2, "thr": thr
                })

                print(f"Rep {rep+1} Fold {fold:02d} | thr={thr:.2f} | PR-AUC={pr_auc:.3f} | ROC-AUC={roc_auc:.3f} | "
                      f"Brier={brier:.3f} | R={rec:.3f} P={prec:.3f}")

results_df = pd.DataFrame(rows)

summary = (results_df.groupby(["cohort","model","mode"])
           .agg(roc_auc_mean=("roc_auc","mean"),
                pr_auc_mean=("pr_auc","mean"),
                brier_mean=("brier","mean"),
                recall_mean=("recall","mean"),
                precision_mean=("precision","mean"),
                f2_mean=("f2","mean"),
                thr_mean=("thr","mean"),
                folds=("thr","count"))
           .reset_index()
           .sort_values("recall_mean", ascending=False))

print("\n=== SUMMARY ===")
print(summary.to_string(index=False))

# ============================================================
# 🔥 PROXY DETECTOR: single-feature AUC scan
# If any single feature gives AUC ~ 1.0, it's a proxy.
# Trees will exploit it -> perfect results.
# ============================================================

def safe_auc(y_true, scores):
    if pd.Series(y_true).nunique() < 2:
        return np.nan
    if np.nanstd(scores) < 1e-12:
        return np.nan
    return roc_auc_score(y_true, scores)

X_fill = X.copy().apply(pd.to_numeric, errors="coerce")
X_fill = X_fill.fillna(X_fill.median(numeric_only=True))

proxy_scores = []
for col in X_fill.columns:
    s = X_fill[col].values.astype(float)
    auc_pos = safe_auc(y, s)
    auc_neg = safe_auc(y, -s)
    best_auc = np.nanmax([auc_pos, auc_neg])
    proxy_scores.append((col, best_auc, float(np.nanstd(s)), int(X_fill[col].nunique())))

proxy_df = pd.DataFrame(proxy_scores, columns=["feature", "best_single_auc", "std", "n_unique"])
proxy_df = proxy_df.sort_values("best_single_auc", ascending=False)

print("\n[PROXY DETECTOR] Top 15 single-feature AUCs:")
print(proxy_df.head(15).to_string(index=False))

sus = proxy_df[proxy_df["best_single_auc"] >= 0.98]
print(f"\n[PROXY DETECTOR] Features with single-feature AUC >= 0.98: {len(sus)}")
if len(sus):
    print(sus.head(30).to_string(index=False))

Loaded: antepartum shape: (1430, 22)
Sample columns: ['match_key', 'antepartum__depression', 'age', 'epds_antepartum', 'anxious_a', 'avoidant_a', 'reframing', 'selfdistraction', 'venting', 'use_of_informational', 'active', 'denial', 'religion', 'humour', 'disengagement']
Detected EPDS: epds_antepartum | cutoff: 12
y distribution: Counter({0: 1196, 1: 234})

Selected columns (most populated per concept):
  age                          -> age                                  non-missing=1332
  anxious_attachment           -> anxious_a                            non-missing=1430
  avoidant_attachment          -> avoidant_a                           non-missing=1430
  reframing                    -> reframing                            non-missing=1430
  selfdistraction              -> selfdistraction                      non-missing=1332
  venting                      -> venting                              non-missing=1430
  use_of_informational_support -> use_of_informational           

# 3

In [3]:
# 3============================================================
# ✅ FINAL: Block-consistent DATA FIX + Group-safe CV + PR-AUC + Brier + Proxy checks
# ------------------------------------------------------------
# Fixes your true issue:
# - You have TWO column blocks in the Excel (one filled for ~234 rows, one filled for ~1430 rows)
# - Mixing columns across blocks leaks "which block" -> perfect trees
#
# This script:
# ✅ Detects EPDS and creates y
# ✅ Builds a clean numeric DF
# ✅ Creates TWO candidate feature blocks (FULL vs SHORT) based on non-missing rate
# ✅ Picks ONE block that:
#     - keeps both classes
#     - reduces missingness gap by class
# ✅ Group-safe CV + inner-val thresholding
# ✅ Shuffle sanity
# ✅ Proxy detector (single-feature)
# ============================================================

import warnings, re, hashlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from collections import Counter

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.calibration import CalibratedClassifierCV

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    precision_score, recall_score, f1_score, fbeta_score, confusion_matrix, make_scorer
)
from sklearn.model_selection import RandomizedSearchCV

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# Optional XGBoost
use_xgboost = False
try:
    from xgboost import XGBClassifier
    use_xgboost = True
except Exception:
    use_xgboost = False

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# -------------------------------
# 0) CONFIG
# -------------------------------
COHORT = "antepartum"   # "antepartum" or "postpartum"
ANTE_PATH = "/kaggle/input/datasets/maishaidaralkhateeb/ppd-depression/Antepartum_Merged_from_Postpartum.xlsx"
POST_PATH = "/kaggle/input/ppd-depression/Postpartum Depression.xlsx"
EPDS_CUTOFFS = {"antepartum": 12, "postpartum": 9}
cutoff = EPDS_CUTOFFS[COHORT]

# name-based leakage proxies (keep epds only for y, then drop from X)
LEAK_PATTERNS = ["depress", "depression", "without", "outcome", "label"]

# block detection thresholds
FULL_BLOCK_MIN_RATE = 0.70   # columns present in >=70% rows
SHORT_BLOCK_MAX_RATE = 0.30  # columns present in <=30% rows

# CV settings
N_SPLITS = 5
N_REPEATS = 3
INNER_VAL_FRACTION = 0.20
BETA = 2.0

# -------------------------------
# 1) Load + clean headers
# -------------------------------
def load_and_clean_excel(path: str) -> pd.DataFrame:
    df = pd.read_excel(path, header=0)
    df = df.dropna(axis=1, how="all")
    df.columns = (
        df.columns.astype(str)
        .str.strip().str.lower()
        .str.replace(" ", "_")
        .str.replace(r"[^0-9a-zA-Z_]", "", regex=True)
    )
    df = df.loc[:, ~df.columns.duplicated()].copy()
    df = df.drop_duplicates()
    return df

df = load_and_clean_excel(ANTE_PATH if COHORT == "antepartum" else POST_PATH)
print("Loaded:", COHORT, "shape:", df.shape)
print("Sample columns:", df.columns.tolist()[:15])

# -------------------------------
# 2) Detect EPDS column for label
# -------------------------------
def find_best_match(cols, keywords):
    for kw in keywords:
        hits = [c for c in cols if kw in c]
        if hits:
            return hits[0]
    return None

if COHORT == "antepartum":
    epds_col = find_best_match(df.columns, ["epds_antepartum", "epdsantepartum", "epds"])
else:
    epds_col = find_best_match(df.columns, ["epds_postpartum", "epdspostpartum", "epds"])

if epds_col is None:
    raise ValueError("Could not find EPDS column.")

df[epds_col] = pd.to_numeric(df[epds_col], errors="coerce")
y = (df[epds_col] >= cutoff).astype(int)

print("Detected EPDS:", epds_col, "| cutoff:", cutoff)
print("y distribution:", Counter(y))

# -------------------------------
# 3) Build numeric-only candidate feature table
# -------------------------------
# Drop EPDS and obvious leakage-proxy columns from features
drop_cols = [c for c in df.columns if any(p in c for p in LEAK_PATTERNS)]
drop_cols += [c for c in df.columns if "epds" in c]  # remove epds variants from X

X_all = df.drop(columns=drop_cols, errors="ignore").copy()
for c in X_all.columns:
    X_all[c] = pd.to_numeric(X_all[c], errors="coerce")

# drop all-empty cols
X_all = X_all.dropna(axis=1, how="all")

# keep only rows where at least something is present
mask = X_all.notna().any(axis=1) & y.notna()
X_all = X_all.loc[mask].copy()
y = y.loc[mask].copy()

print("\nNumeric feature pool shape:", X_all.shape)
print("y distribution after basic align:", Counter(y))

# -------------------------------
# 4) Identify blocks by coverage (FULL vs SHORT)
# -------------------------------
non_missing_rate = X_all.notna().mean().sort_values(ascending=False)

full_cols = non_missing_rate[non_missing_rate >= FULL_BLOCK_MIN_RATE].index.tolist()
short_cols = non_missing_rate[non_missing_rate <= SHORT_BLOCK_MAX_RATE].index.tolist()

print("\n[BLOCK DETECT]")
print("Full block cols:", len(full_cols), "| Short block cols:", len(short_cols))

X_full = X_all[full_cols].copy() if len(full_cols) else pd.DataFrame(index=X_all.index)
X_short = X_all[short_cols].copy() if len(short_cols) else pd.DataFrame(index=X_all.index)

# -------------------------------
# 5) Choose ONE block (don’t mix)
# Criteria:
#  - both classes remain
#  - missingness gap by class is smaller (less “missingness = label”)
# -------------------------------
def missing_gap_score(X, y):
    if X.shape[1] == 0:
        return np.inf
    miss_by_class = X.isna().groupby(y).mean()
    # if class missing -> invalid
    if miss_by_class.shape[0] < 2:
        return np.inf
    gap = (miss_by_class.loc[1] - miss_by_class.loc[0]).abs()
    return float(gap.mean())

def block_valid(X, y):
    # keep rows with at least 3 values present to avoid totally empty rows
    keep = X.notna().sum(axis=1) >= min(3, X.shape[1])
    X2 = X.loc[keep].copy()
    y2 = y.loc[keep].copy()
    return X2, y2, (y2.nunique() == 2)

Xf, yf, okf = block_valid(X_full, y)
Xs, ys, oks = block_valid(X_short, y)

gap_f = missing_gap_score(Xf, yf) if okf else np.inf
gap_s = missing_gap_score(Xs, ys) if oks else np.inf

print("\n[BLOCK SCORES]")
print("FULL  -> valid:", okf, "| rows:", len(Xf), "| gap_mean:", gap_f)
print("SHORT -> valid:", oks, "| rows:", len(Xs), "| gap_mean:", gap_s)

# choose block with smaller missingness-gap score
if gap_f <= gap_s:
    X, y = Xf, yf
    CHOSEN_BLOCK = "FULL"
else:
    X, y = Xs, ys
    CHOSEN_BLOCK = "SHORT"

print(f"\n✅ Chosen block: {CHOSEN_BLOCK}")
print("Final X shape:", X.shape)
print("Final y distribution:", Counter(y))

if y.nunique() < 2:
    raise ValueError("After block selection, only one class remains. The sheet is inconsistent.")

# show missingness gap top 10
overall_miss = X.isna().mean().sort_values(ascending=False)
miss_by_class = X.isna().groupby(y).mean()
gap = (miss_by_class.loc[1] - miss_by_class.loc[0]).abs().sort_values(ascending=False)
print("\nMissingness gap top 10 (should not be ~1.0 everywhere):")
print(gap.head(10))

# -------------------------------
# 6) Build groups (hash of feature rows) — still good practice
# -------------------------------
X_for_hash = X.fillna("__MISSING__").astype(str)
groups = X_for_hash.apply(lambda r: hashlib.md5("||".join(r.values.tolist()).encode("utf-8")).hexdigest(), axis=1)
groups = pd.Series(groups, index=X.index)
print("\n[GROUP CHECK] Unique groups:", groups.nunique(), "out of", len(groups))
print("[GROUP CHECK] Largest group size:", int(groups.value_counts().max()))

# -------------------------------
# 7) Splitters
# -------------------------------
try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAS_SGKF = True
except Exception:
    HAS_SGKF = False
    from sklearn.model_selection import GroupKFold

def outer_splitter(seed: int):
    if HAS_SGKF:
        cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
        return cv.split(X, y, groups=groups)
    else:
        gkf = GroupKFold(n_splits=N_SPLITS)
        return gkf.split(X, y, groups=groups)

def group_safe_inner_split(train_idx, val_frac, seed):
    rng = np.random.default_rng(seed)
    train_groups = groups.iloc[train_idx]
    uniq_g = train_groups.unique()
    rng.shuffle(uniq_g)
    n_val_groups = max(1, int(len(uniq_g) * val_frac))
    val_g = set(uniq_g[:n_val_groups])
    inner_val_mask = train_groups.isin(val_g).values
    inner_val_idx = train_idx[inner_val_mask]
    inner_train_idx = train_idx[~inner_val_mask]
    return inner_train_idx, inner_val_idx

# -------------------------------
# 8) Models + pipelines
# -------------------------------
def make_base_estimator(model_name, mode, spw):
    if model_name == "LR":
        cw = "balanced" if mode == "weighted" else None
        return LogisticRegression(max_iter=5000, class_weight=cw, random_state=RANDOM_STATE)
    if model_name == "SVM":
        cw = "balanced" if mode == "weighted" else None
        return SVC(kernel="rbf", probability=True, class_weight=cw, random_state=RANDOM_STATE)
    if model_name == "RF":
        cw = "balanced" if mode == "weighted" else None
        return RandomForestClassifier(n_estimators=400, class_weight=cw, random_state=RANDOM_STATE, n_jobs=-1)
    if model_name == "XGB":
        if not use_xgboost:
            return HistGradientBoostingClassifier(random_state=RANDOM_STATE)
        spw2 = spw if mode == "weighted" else 1.0
        return XGBClassifier(
            n_estimators=500, max_depth=3, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
            scale_pos_weight=spw2, eval_metric="logloss",
            random_state=RANDOM_STATE, n_jobs=-1
        )
    return HistGradientBoostingClassifier(random_state=RANDOM_STATE)

def build_pipeline(model_name, mode, spw):
    imputer = ("imputer", SimpleImputer(strategy="median"))
    scaler = ("scaler", StandardScaler())
    estimator = ("model", make_base_estimator(model_name, mode, spw))
    if mode == "resampled":
        smote = ("smote", SMOTE(sampling_strategy=0.8, random_state=RANDOM_STATE))
        under = ("under", RandomUnderSampler(sampling_strategy=1.0, random_state=RANDOM_STATE))
        return ImbPipeline([imputer, scaler, smote, under, estimator])
    return SkPipeline([imputer, scaler, estimator])

def get_param_distributions(model_name):
    if model_name == "LR":
        return {"model__C": np.logspace(-3, 3, 25)}
    if model_name == "SVM":
        return {"model__C": np.logspace(-2, 3, 20), "model__gamma": ["scale", "auto"]}
    if model_name == "RF":
        return {"model__max_depth": [None, 4, 7, 12], "model__min_samples_leaf": [1, 2, 4]}
    if model_name == "XGB":
        return {"model__max_depth": [2, 3, 4], "model__learning_rate": [0.03, 0.05, 0.1]}
    return {}

def choose_threshold_on_val(y_val, p_val, beta=BETA):
    thresholds = np.linspace(0.05, 0.95, 91)
    best_t, best_score = 0.5, -1.0
    for t in thresholds:
        y_hat = (p_val >= t).astype(int)
        score = fbeta_score(y_val, y_hat, beta=beta, zero_division=0)
        if score > best_score:
            best_score = score
            best_t = t
    return best_t, best_score

# -------------------------------
# 9) Label-shuffle sanity test
# -------------------------------
def quick_groupcv_auc(seed=42, shuffle_labels=False):
    rng = np.random.default_rng(seed)
    y_tested = y.copy()
    if shuffle_labels:
        y_tested = pd.Series(rng.permutation(y_tested.values), index=y_tested.index)

    aucs, aps = [], []
    for tr_idx, te_idx in outer_splitter(seed):
        X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
        y_tr, y_te = y_tested.iloc[tr_idx], y_tested.iloc[te_idx]
        n_pos = int((y_tr == 1).sum()); n_neg = int((y_tr == 0).sum())
        spw = (n_neg / max(n_pos, 1))
        pipe = build_pipeline("SVM", "weighted", spw)
        pipe.fit(X_tr, y_tr)
        p = pipe.predict_proba(X_te)[:, 1]
        aucs.append(roc_auc_score(y_te, p))
        aps.append(average_precision_score(y_te, p))
    return float(np.mean(aucs)), float(np.mean(aps))

print("\n[SHUFFLE SANITY]")
ra, rp = quick_groupcv_auc(seed=42, shuffle_labels=False)
sa, sp = quick_groupcv_auc(seed=42, shuffle_labels=True)
print(f"Real     -> ROC-AUC={ra:.3f}, PR-AUC={rp:.3f}")
print(f"Shuffled -> ROC-AUC={sa:.3f}, PR-AUC={sp:.3f}")

# -------------------------------
# 10) Full evaluation
# -------------------------------
model_specs = ["LR", "SVM", "RF", "XGB"]
modes = ["weighted", "resampled"]
f2_scorer = make_scorer(fbeta_score, beta=BETA, zero_division=0)

rows = []
for rep in range(N_REPEATS):
    rep_seed = RANDOM_STATE + rep
    print(f"\n===== REPEAT {rep+1}/{N_REPEATS} | seed={rep_seed} =====")

    for model_name in model_specs:
        for mode in modes:
            print(f"\n=== {COHORT.upper()} | MODEL={model_name} | MODE={mode} ===")

            for fold, (train_idx, test_idx) in enumerate(outer_splitter(rep_seed), start=1):
                X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
                y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

                n_pos = int((y_train == 1).sum()); n_neg = int((y_train == 0).sum())
                spw = (n_neg / max(n_pos, 1))

                inner_train_idx, inner_val_idx = group_safe_inner_split(train_idx, INNER_VAL_FRACTION, seed=rep_seed + 1000 + fold)
                X_tr, y_tr = X.iloc[inner_train_idx], y.iloc[inner_train_idx]
                X_val, y_val = X.iloc[inner_val_idx], y.iloc[inner_val_idx]

                pipe = build_pipeline(model_name, mode, spw)
                param_dist = get_param_distributions(model_name)

                search = RandomizedSearchCV(
                    estimator=pipe,
                    param_distributions=param_dist,
                    n_iter=15,
                    scoring=f2_scorer,
                    cv=3,
                    random_state=rep_seed,
                    n_jobs=-1,
                    refit=True
                )
                search.fit(X_tr, y_tr)
                best_pipe = search.best_estimator_

                cal = CalibratedClassifierCV(best_pipe, method="sigmoid", cv=3)
                cal.fit(X_tr, y_tr)

                p_val = cal.predict_proba(X_val)[:, 1]
                thr, _ = choose_threshold_on_val(y_val.values, p_val, beta=BETA)

                p_test = cal.predict_proba(X_test)[:, 1]
                y_hat = (p_test >= thr).astype(int)

                roc_auc = roc_auc_score(y_test, p_test)
                pr_auc = average_precision_score(y_test, p_test)
                brier = brier_score_loss(y_test, p_test)

                prec = precision_score(y_test, y_hat, zero_division=0)
                rec = recall_score(y_test, y_hat, zero_division=0)
                f1 = f1_score(y_test, y_hat, zero_division=0)
                f2 = fbeta_score(y_test, y_hat, beta=BETA, zero_division=0)

                tn, fp, fn, tp = confusion_matrix(y_test, y_hat).ravel()
                spec = tn / (tn + fp) if (tn + fp) else 0.0

                rows.append({
                    "cohort": COHORT, "block": CHOSEN_BLOCK, "repeat": rep+1, "fold": fold,
                    "model": model_name, "mode": mode,
                    "roc_auc": roc_auc, "pr_auc": pr_auc, "brier": brier,
                    "precision": prec, "recall": rec, "specificity": spec,
                    "f1": f1, "f2": f2, "thr": thr
                })

                print(f"Rep {rep+1} Fold {fold:02d} | thr={thr:.2f} | PR-AUC={pr_auc:.3f} | ROC-AUC={roc_auc:.3f} | "
                      f"Brier={brier:.3f} | R={rec:.3f} P={prec:.3f}")

results_df = pd.DataFrame(rows)
summary = (results_df.groupby(["cohort","block","model","mode"])
           .agg(roc_auc_mean=("roc_auc","mean"),
                pr_auc_mean=("pr_auc","mean"),
                brier_mean=("brier","mean"),
                recall_mean=("recall","mean"),
                precision_mean=("precision","mean"),
                f2_mean=("f2","mean"),
                thr_mean=("thr","mean"),
                folds=("thr","count"))
           .reset_index()
           .sort_values("recall_mean", ascending=False))

print("\n=== SUMMARY ===")
print(summary.to_string(index=False))

# -------------------------------
# 11) Proxy detector (single feature)
# -------------------------------
def safe_auc(y_true, scores):
    if pd.Series(y_true).nunique() < 2:
        return np.nan
    if np.nanstd(scores) < 1e-12:
        return np.nan
    return roc_auc_score(y_true, scores)

X_fill = X.copy().apply(pd.to_numeric, errors="coerce")
X_fill = X_fill.fillna(X_fill.median(numeric_only=True))

proxy_scores = []
for col in X_fill.columns:
    s = X_fill[col].values.astype(float)
    auc_pos = safe_auc(y, s)
    auc_neg = safe_auc(y, -s)
    best_auc = np.nanmax([auc_pos, auc_neg])
    proxy_scores.append((col, best_auc, float(np.nanstd(s)), int(X_fill[col].nunique())))

proxy_df = pd.DataFrame(proxy_scores, columns=["feature", "best_single_auc", "std", "n_unique"])
proxy_df = proxy_df.sort_values("best_single_auc", ascending=False)

print("\n[PROXY DETECTOR] Top 15 single-feature AUCs:")
print(proxy_df.head(15).to_string(index=False))

sus = proxy_df[proxy_df["best_single_auc"] >= 0.98]
print(f"\n[PROXY DETECTOR] Features with single-feature AUC >= 0.98: {len(sus)}")
if len(sus):
    print(sus.head(30).to_string(index=False))

Loaded: antepartum shape: (1430, 22)
Sample columns: ['match_key', 'antepartum__depression', 'age', 'epds_antepartum', 'anxious_a', 'avoidant_a', 'reframing', 'selfdistraction', 'venting', 'use_of_informational', 'active', 'denial', 'religion', 'humour', 'disengagement']
Detected EPDS: epds_antepartum | cutoff: 12
y distribution: Counter({0: 1196, 1: 234})

Numeric feature pool shape: (1430, 17)
y distribution after basic align: Counter({0: 1196, 1: 234})

[BLOCK DETECT]
Full block cols: 17 | Short block cols: 0

[BLOCK SCORES]
FULL  -> valid: True | rows: 1430 | gap_mean: 0.01932913633680897
SHORT -> valid: True | rows: 1430 | gap_mean: inf

✅ Chosen block: FULL
Final X shape: (1430, 17)
Final y distribution: Counter({0: 1196, 1: 234})

Missingness gap top 10 (should not be ~1.0 everywhere):
age                0.081940
emotional_s        0.081940
selfdistraction    0.081940
substance          0.081940
selfblame          0.000836
anxious_a          0.000000
planning           0.000000


# 4

In [4]:
# 3.2============================================================
# ✅ FINAL Kaggle Code (Block-consistent + Group-safe CV)
# - Label from EPDS cutoff (no OUTCOME)
# - Drop leakage proxies by name
# - Choose ONE feature block (FULL vs SHORT) by coverage
# - Group-safe CV + inner-val thresholding (NO test tuning)
# - Randomized search + calibration (sigmoid)
# - ROC-AUC, PR-AUC, Brier
# - Recall-first threshold with precision constraint (paper-friendly)
# - Shuffle sanity test
# ============================================================

import warnings, re, hashlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from collections import Counter

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.calibration import CalibratedClassifierCV

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    precision_score, recall_score, f1_score, fbeta_score, confusion_matrix, make_scorer
)
from sklearn.model_selection import RandomizedSearchCV

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# Optional XGBoost
use_xgboost = False
try:
    from xgboost import XGBClassifier
    use_xgboost = True
except Exception:
    use_xgboost = False

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# -------------------------------
# 0) CONFIG
# -------------------------------
COHORT = "antepartum"   # "antepartum" or "postpartum"
ANTE_PATH = "/kaggle/input/datasets/maishaidaralkhateeb/ppd-depression/Antepartum_Merged_from_Postpartum.xlsx"
POST_PATH = "/kaggle/input/ppd-depression/Postpartum Depression.xlsx"
EPDS_CUTOFFS = {"antepartum": 12, "postpartum": 9}
cutoff = EPDS_CUTOFFS[COHORT]

LEAK_PATTERNS = ["depress", "depression", "without", "outcome", "label"]

# block detection thresholds
FULL_BLOCK_MIN_RATE = 0.70
SHORT_BLOCK_MAX_RATE = 0.30

# CV settings
N_SPLITS = 5
N_REPEATS = 3
INNER_VAL_FRACTION = 0.20

# Decision policy (paper-friendly)
TARGET_RECALL = 0.85         # tune: 0.80–0.95 depending on safety goals
MIN_PRECISION = 0.20         # tune: 0.15–0.35 depending on alert tolerance
BETA_FALLBACK = 2.0          # if constraint can't be met, fallback to F2

# -------------------------------
# 1) Load + clean headers
# -------------------------------
def load_and_clean_excel(path: str) -> pd.DataFrame:
    df = pd.read_excel(path, header=0)
    df = df.dropna(axis=1, how="all")
    df.columns = (
        df.columns.astype(str)
        .str.strip().str.lower()
        .str.replace(" ", "_")
        .str.replace(r"[^0-9a-zA-Z_]", "", regex=True)
    )
    df = df.loc[:, ~df.columns.duplicated()].copy()
    df = df.drop_duplicates()
    return df

df = load_and_clean_excel(ANTE_PATH if COHORT == "antepartum" else POST_PATH)
print("Loaded:", COHORT, "shape:", df.shape)
print("Sample columns:", df.columns.tolist()[:15])

# -------------------------------
# 2) Detect EPDS column for label
# -------------------------------
def find_best_match(cols, keywords):
    for kw in keywords:
        hits = [c for c in cols if kw in c]
        if hits:
            return hits[0]
    return None

if COHORT == "antepartum":
    epds_col = find_best_match(df.columns, ["epds_antepartum", "epdsantepartum", "epds"])
else:
    epds_col = find_best_match(df.columns, ["epds_postpartum", "epdspostpartum", "epds"])

if epds_col is None:
    raise ValueError("Could not find EPDS column.")

df[epds_col] = pd.to_numeric(df[epds_col], errors="coerce")
y = (df[epds_col] >= cutoff).astype(int)

print("Detected EPDS:", epds_col, "| cutoff:", cutoff)
print("y distribution:", Counter(y))
print("PR-AUC baseline (prevalence):", round(y.mean(), 3))

# -------------------------------
# 3) Build numeric-only candidate feature table
# -------------------------------
drop_cols = [c for c in df.columns if any(p in c for p in LEAK_PATTERNS)]
drop_cols += [c for c in df.columns if "epds" in c]  # remove epds variants from X

X_all = df.drop(columns=drop_cols, errors="ignore").copy()
for c in X_all.columns:
    X_all[c] = pd.to_numeric(X_all[c], errors="coerce")
X_all = X_all.dropna(axis=1, how="all")

mask = X_all.notna().any(axis=1) & y.notna()
X_all = X_all.loc[mask].copy()
y = y.loc[mask].copy()

print("\nNumeric feature pool shape:", X_all.shape)
print("y distribution after basic align:", Counter(y))

# -------------------------------
# 4) Identify blocks by coverage (FULL vs SHORT)
# -------------------------------
non_missing_rate = X_all.notna().mean().sort_values(ascending=False)
full_cols = non_missing_rate[non_missing_rate >= FULL_BLOCK_MIN_RATE].index.tolist()
short_cols = non_missing_rate[non_missing_rate <= SHORT_BLOCK_MAX_RATE].index.tolist()

print("\n[BLOCK DETECT] Full block cols:", len(full_cols), "| Short block cols:", len(short_cols))

X_full = X_all[full_cols].copy() if len(full_cols) else pd.DataFrame(index=X_all.index)
X_short = X_all[short_cols].copy() if len(short_cols) else pd.DataFrame(index=X_all.index)

def block_valid(X, y):
    keep = X.notna().sum(axis=1) >= min(3, X.shape[1])
    X2 = X.loc[keep].copy()
    y2 = y.loc[keep].copy()
    return X2, y2, (y2.nunique() == 2)

def missing_gap_score(X, y):
    if X.shape[1] == 0: return np.inf
    miss_by_class = X.isna().groupby(y).mean()
    if miss_by_class.shape[0] < 2: return np.inf
    gap = (miss_by_class.loc[1] - miss_by_class.loc[0]).abs()
    return float(gap.mean())

Xf, yf, okf = block_valid(X_full, y)
Xs, ys, oks = block_valid(X_short, y)

gap_f = missing_gap_score(Xf, yf) if okf else np.inf
gap_s = missing_gap_score(Xs, ys) if oks else np.inf

print("\n[BLOCK SCORES]")
print("FULL  -> valid:", okf, "| rows:", len(Xf), "| gap_mean:", gap_f)
print("SHORT -> valid:", oks, "| rows:", len(Xs), "| gap_mean:", gap_s)

if gap_f <= gap_s:
    X, y = Xf, yf
    CHOSEN_BLOCK = "FULL"
else:
    X, y = Xs, ys
    CHOSEN_BLOCK = "SHORT"

print(f"\n✅ Chosen block: {CHOSEN_BLOCK}")
print("Final X shape:", X.shape)
print("Final y distribution:", Counter(y))

# -------------------------------
# 5) Groups by feature-row hash
# -------------------------------
X_for_hash = X.fillna("__MISSING__").astype(str)
groups = X_for_hash.apply(lambda r: hashlib.md5("||".join(r.values.tolist()).encode("utf-8")).hexdigest(), axis=1)
groups = pd.Series(groups, index=X.index)

print("\n[GROUP CHECK] Unique groups:", groups.nunique(), "out of", len(groups))
print("[GROUP CHECK] Largest group size:", int(groups.value_counts().max()))

# -------------------------------
# 6) Splitters
# -------------------------------
try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAS_SGKF = True
except Exception:
    HAS_SGKF = False
    from sklearn.model_selection import GroupKFold

def outer_splitter(seed: int):
    if HAS_SGKF:
        cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
        return cv.split(X, y, groups=groups)
    else:
        gkf = GroupKFold(n_splits=N_SPLITS)
        return gkf.split(X, y, groups=groups)

def group_safe_inner_split(train_idx, val_frac, seed):
    rng = np.random.default_rng(seed)
    train_groups = groups.iloc[train_idx]
    uniq_g = train_groups.unique()
    rng.shuffle(uniq_g)
    n_val_groups = max(1, int(len(uniq_g) * val_frac))
    val_g = set(uniq_g[:n_val_groups])
    inner_val_mask = train_groups.isin(val_g).values
    inner_val_idx = train_idx[inner_val_mask]
    inner_train_idx = train_idx[~inner_val_mask]
    return inner_train_idx, inner_val_idx

# -------------------------------
# 7) Models + pipelines
# -------------------------------
def make_base_estimator(model_name, mode, spw):
    if model_name == "LR":
        cw = "balanced" if mode == "weighted" else None
        return LogisticRegression(max_iter=5000, class_weight=cw, random_state=RANDOM_STATE)
    if model_name == "SVM":
        cw = "balanced" if mode == "weighted" else None
        return SVC(kernel="rbf", probability=True, class_weight=cw, random_state=RANDOM_STATE)
    if model_name == "RF":
        cw = "balanced" if mode == "weighted" else None
        return RandomForestClassifier(n_estimators=400, class_weight=cw, random_state=RANDOM_STATE, n_jobs=-1)
    if model_name == "XGB":
        if not use_xgboost:
            return HistGradientBoostingClassifier(random_state=RANDOM_STATE)
        spw2 = spw if mode == "weighted" else 1.0
        return XGBClassifier(
            n_estimators=500, max_depth=3, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
            scale_pos_weight=spw2, eval_metric="logloss",
            random_state=RANDOM_STATE, n_jobs=-1
        )
    return HistGradientBoostingClassifier(random_state=RANDOM_STATE)

def build_pipeline(model_name, mode, spw):
    imputer = ("imputer", SimpleImputer(strategy="median"))
    scaler = ("scaler", StandardScaler())
    estimator = ("model", make_base_estimator(model_name, mode, spw))
    if mode == "resampled":
        smote = ("smote", SMOTE(sampling_strategy=0.8, random_state=RANDOM_STATE))
        under = ("under", RandomUnderSampler(sampling_strategy=1.0, random_state=RANDOM_STATE))
        return ImbPipeline([imputer, scaler, smote, under, estimator])
    return SkPipeline([imputer, scaler, estimator])

def get_param_distributions(model_name):
    if model_name == "LR":
        return {"model__C": np.logspace(-3, 3, 25)}
    if model_name == "SVM":
        return {"model__C": np.logspace(-2, 3, 20), "model__gamma": ["scale", "auto"]}
    if model_name == "RF":
        return {"model__max_depth": [None, 4, 7, 12], "model__min_samples_leaf": [1, 2, 4]}
    if model_name == "XGB":
        return {"model__max_depth": [2, 3, 4], "model__learning_rate": [0.03, 0.05, 0.1]}
    return {}

# -------------------------------
# 8) Threshold selection (recall-first + precision floor)
# -------------------------------
def choose_threshold_recall_first(y_val, p_val, target_recall=TARGET_RECALL, min_precision=MIN_PRECISION, beta_fallback=BETA_FALLBACK):
    thresholds = np.linspace(0.01, 0.99, 99)
    best = None  # (recall, precision, f1, thr)
    for t in thresholds:
        y_hat = (p_val >= t).astype(int)
        rec = recall_score(y_val, y_hat, zero_division=0)
        prec = precision_score(y_val, y_hat, zero_division=0)
        if rec >= target_recall and prec >= min_precision:
            f1 = f1_score(y_val, y_hat, zero_division=0)
            cand = (rec, prec, f1, t)
            if best is None or cand > best:
                best = cand

    if best is not None:
        rec, prec, f1, t = best
        return float(t), {"policy": "recall+precision", "val_recall": rec, "val_precision": prec, "val_f1": f1}

    # fallback: maximize F-beta
    best_t, best_score = 0.5, -1.0
    for t in thresholds:
        y_hat = (p_val >= t).astype(int)
        score = fbeta_score(y_val, y_hat, beta=beta_fallback, zero_division=0)
        if score > best_score:
            best_score = score
            best_t = t
    return float(best_t), {"policy": f"fallback_f{beta_fallback}", "val_fbeta": best_score}

# -------------------------------
# 9) Shuffle sanity test (quick)
# -------------------------------
def quick_groupcv_auc(seed=42, shuffle_labels=False):
    rng = np.random.default_rng(seed)
    y_tested = y.copy()
    if shuffle_labels:
        y_tested = pd.Series(rng.permutation(y_tested.values), index=y_tested.index)

    aucs, aps = [], []
    for tr_idx, te_idx in outer_splitter(seed):
        X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
        y_tr, y_te = y_tested.iloc[tr_idx], y_tested.iloc[te_idx]
        n_pos = int((y_tr == 1).sum()); n_neg = int((y_tr == 0).sum())
        spw = (n_neg / max(n_pos, 1))
        pipe = build_pipeline("LR", "weighted", spw)  # fast sanity baseline
        pipe.fit(X_tr, y_tr)
        p = pipe.predict_proba(X_te)[:, 1]
        aucs.append(roc_auc_score(y_te, p))
        aps.append(average_precision_score(y_te, p))
    return float(np.mean(aucs)), float(np.mean(aps))

print("\n[SHUFFLE SANITY]")
ra, rp = quick_groupcv_auc(seed=42, shuffle_labels=False)
sa, sp = quick_groupcv_auc(seed=42, shuffle_labels=True)
print(f"Real     -> ROC-AUC={ra:.3f}, PR-AUC={rp:.3f}")
print(f"Shuffled -> ROC-AUC={sa:.3f}, PR-AUC={sp:.3f}")

# -------------------------------
# 10) Full evaluation
# -------------------------------
model_specs = ["LR", "SVM", "RF", "XGB"]
modes = ["weighted", "resampled"]
f2_scorer = make_scorer(fbeta_score, beta=2.0, zero_division=0)

rows = []
for rep in range(N_REPEATS):
    rep_seed = RANDOM_STATE + rep
    print(f"\n===== REPEAT {rep+1}/{N_REPEATS} | seed={rep_seed} =====")

    for model_name in model_specs:
        for mode in modes:
            print(f"\n=== {COHORT.upper()} | MODEL={model_name} | MODE={mode} ===")

            for fold, (train_idx, test_idx) in enumerate(outer_splitter(rep_seed), start=1):
                X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
                y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
                n_pos = int((y_train == 1).sum()); n_neg = int((y_train == 0).sum())
                spw = (n_neg / max(n_pos, 1))

                inner_train_idx, inner_val_idx = group_safe_inner_split(train_idx, INNER_VAL_FRACTION, seed=rep_seed + 1000 + fold)
                X_tr, y_tr = X.iloc[inner_train_idx], y.iloc[inner_train_idx]
                X_val, y_val = X.iloc[inner_val_idx], y.iloc[inner_val_idx]

                pipe = build_pipeline(model_name, mode, spw)
                param_dist = get_param_distributions(model_name)

                search = RandomizedSearchCV(
                    estimator=pipe,
                    param_distributions=param_dist,
                    n_iter=15,
                    scoring=f2_scorer,
                    cv=3,
                    random_state=rep_seed,
                    n_jobs=-1,
                    refit=True
                )
                search.fit(X_tr, y_tr)
                best_pipe = search.best_estimator_

                cal = CalibratedClassifierCV(best_pipe, method="sigmoid", cv=3)
                cal.fit(X_tr, y_tr)

                p_val = cal.predict_proba(X_val)[:, 1]
                thr, thr_info = choose_threshold_recall_first(y_val.values, p_val)

                p_test = cal.predict_proba(X_test)[:, 1]
                y_hat = (p_test >= thr).astype(int)

                roc_auc = roc_auc_score(y_test, p_test)
                pr_auc = average_precision_score(y_test, p_test)
                brier = brier_score_loss(y_test, p_test)

                prec = precision_score(y_test, y_hat, zero_division=0)
                rec = recall_score(y_test, y_hat, zero_division=0)
                f1 = f1_score(y_test, y_hat, zero_division=0)
                f2 = fbeta_score(y_test, y_hat, beta=2.0, zero_division=0)

                tn, fp, fn, tp = confusion_matrix(y_test, y_hat).ravel()
                spec = tn / (tn + fp) if (tn + fp) else 0.0

                rows.append({
                    "cohort": COHORT, "block": CHOSEN_BLOCK, "repeat": rep+1, "fold": fold,
                    "model": model_name, "mode": mode,
                    "roc_auc": roc_auc, "pr_auc": pr_auc, "brier": brier,
                    "precision": prec, "recall": rec, "specificity": spec,
                    "f1": f1, "f2": f2, "thr": thr,
                    "thr_policy": thr_info.get("policy", "unknown")
                })

                print(f"Rep {rep+1} Fold {fold:02d} | thr={thr:.2f} ({thr_info.get('policy')}) | "
                      f"PR-AUC={pr_auc:.3f} ROC-AUC={roc_auc:.3f} Brier={brier:.3f} | R={rec:.3f} P={prec:.3f}")

results_df = pd.DataFrame(rows)
summary = (results_df.groupby(["cohort","block","model","mode"])
           .agg(roc_auc_mean=("roc_auc","mean"),
                pr_auc_mean=("pr_auc","mean"),
                brier_mean=("brier","mean"),
                recall_mean=("recall","mean"),
                precision_mean=("precision","mean"),
                f2_mean=("f2","mean"),
                thr_mean=("thr","mean"),
                folds=("thr","count"))
           .reset_index()
           .sort_values("pr_auc_mean", ascending=False))

print("\n=== SUMMARY ===")
print(summary.to_string(index=False))

Loaded: antepartum shape: (1430, 22)
Sample columns: ['match_key', 'antepartum__depression', 'age', 'epds_antepartum', 'anxious_a', 'avoidant_a', 'reframing', 'selfdistraction', 'venting', 'use_of_informational', 'active', 'denial', 'religion', 'humour', 'disengagement']
Detected EPDS: epds_antepartum | cutoff: 12
y distribution: Counter({0: 1196, 1: 234})
PR-AUC baseline (prevalence): 0.164

Numeric feature pool shape: (1430, 17)
y distribution after basic align: Counter({0: 1196, 1: 234})

[BLOCK DETECT] Full block cols: 17 | Short block cols: 0

[BLOCK SCORES]
FULL  -> valid: True | rows: 1430 | gap_mean: 0.01932913633680897
SHORT -> valid: True | rows: 1430 | gap_mean: inf

✅ Chosen block: FULL
Final X shape: (1430, 17)
Final y distribution: Counter({0: 1196, 1: 234})

[GROUP CHECK] Unique groups: 1429 out of 1430
[GROUP CHECK] Largest group size: 2

[SHUFFLE SANITY]
Real     -> ROC-AUC=0.793, PR-AUC=0.492
Shuffled -> ROC-AUC=0.487, PR-AUC=0.161

===== REPEAT 1/3 | seed=42 =====

=

# 5

In [5]:
#3  ============================================================
# ✅ FULL FIXED Kaggle Code (Block-consistent + TRUE Group-safe Tuning + Leak-safe Calibration)
# ------------------------------------------------------------
# What’s fixed vs your last version:
# ✅ Block-consistent features (FULL vs SHORT) — still here
# ✅ Outer CV is group-safe (StratifiedGroupKFold when available)
# ✅ RandomizedSearchCV inner CV is NOW group-safe (uses groups!)
# ✅ Calibration is NOW leak-safe:
#    - NO CalibratedClassifierCV(cv=3) leakage
#    - We calibrate using ONLY the held-out inner-val split (Platt scaling)
# ✅ Threshold picked on inner-val only (recall-first + precision floor)
# ✅ ROC-AUC, PR-AUC, Brier on outer test
# ✅ Shuffle sanity test
#
# Notes:
# - If you have a real subject ID column, use it for groups (recommended).
#   Current fallback uses row-hash (good for duplicates, not true subject grouping).
# ============================================================

import warnings, hashlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from collections import Counter

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline as SkPipeline

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    precision_score, recall_score, f1_score, fbeta_score, confusion_matrix, make_scorer
)

from sklearn.model_selection import RandomizedSearchCV

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# Optional XGBoost
use_xgboost = False
try:
    from xgboost import XGBClassifier
    use_xgboost = True
except Exception:
    use_xgboost = False

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# -------------------------------
# 0) CONFIG
# -------------------------------
COHORT = "antepartum"   # "antepartum" or "postpartum"
ANTE_PATH = "/kaggle/input/datasets/maishaidaralkhateeb/ppd-depression/Antepartum_Merged_from_Postpartum.xlsx"
POST_PATH = "/kaggle/input/ppd-depression/Postpartum Depression.xlsx"

EPDS_CUTOFFS = {"antepartum": 12, "postpartum": 9}
cutoff = EPDS_CUTOFFS[COHORT]

# If you have a real ID column, put its cleaned name here, e.g. "participant_id".
# If it exists, we will use it for groups. Otherwise, we fallback to row-hash.
GROUP_ID_COL = None  # e.g. "women_with_and_without_antepartum_depression" if it's truly subject ID

LEAK_PATTERNS = ["depress", "depression", "without", "outcome", "label"]

# block detection thresholds
FULL_BLOCK_MIN_RATE = 0.70
SHORT_BLOCK_MAX_RATE = 0.30

# CV settings
N_SPLITS = 5
N_REPEATS = 3
INNER_VAL_FRACTION = 0.20

# Decision policy (paper-friendly)
TARGET_RECALL = 0.85
MIN_PRECISION = 0.20
BETA_FALLBACK = 2.0

# Search
SEARCH_N_ITER = 15
INNER_SEARCH_SPLITS = 3  # group-safe inner CV splits for RandomizedSearchCV

# -------------------------------
# 1) Load + clean headers
# -------------------------------
def load_and_clean_excel(path: str) -> pd.DataFrame:
    df = pd.read_excel(path, header=0)
    df = df.dropna(axis=1, how="all")
    df.columns = (
        df.columns.astype(str)
        .str.strip().str.lower()
        .str.replace(" ", "_")
        .str.replace(r"[^0-9a-zA-Z_]", "", regex=True)
    )
    df = df.loc[:, ~df.columns.duplicated()].copy()
    df = df.drop_duplicates()
    return df

df = load_and_clean_excel(ANTE_PATH if COHORT == "antepartum" else POST_PATH)
print("Loaded:", COHORT, "shape:", df.shape)
print("Sample columns:", df.columns.tolist()[:15])

# -------------------------------
# 2) Detect EPDS column for label
# -------------------------------
def find_best_match(cols, keywords):
    for kw in keywords:
        hits = [c for c in cols if kw in c]
        if hits:
            return hits[0]
    return None

if COHORT == "antepartum":
    epds_col = find_best_match(df.columns, ["epds_antepartum", "epdsantepartum", "epds"])
else:
    epds_col = find_best_match(df.columns, ["epds_postpartum", "epdspostpartum", "epds"])

if epds_col is None:
    raise ValueError("Could not find EPDS column.")

df[epds_col] = pd.to_numeric(df[epds_col], errors="coerce")
y = (df[epds_col] >= cutoff).astype(int)

print("Detected EPDS:", epds_col, "| cutoff:", cutoff)
print("y distribution:", Counter(y))
print("PR-AUC baseline (prevalence):", round(y.mean(), 3))

# -------------------------------
# 3) Build numeric-only candidate feature table
# -------------------------------
drop_cols = [c for c in df.columns if any(p in c for p in LEAK_PATTERNS)]
drop_cols += [c for c in df.columns if "epds" in c]  # remove EPDS variants from X

X_all = df.drop(columns=drop_cols, errors="ignore").copy()
for c in X_all.columns:
    X_all[c] = pd.to_numeric(X_all[c], errors="coerce")
X_all = X_all.dropna(axis=1, how="all")

mask = X_all.notna().any(axis=1) & y.notna()
X_all = X_all.loc[mask].copy()
y = y.loc[mask].copy()

print("\nNumeric feature pool shape:", X_all.shape)
print("y distribution after basic align:", Counter(y))

# -------------------------------
# 4) Identify blocks by coverage (FULL vs SHORT)
# -------------------------------
non_missing_rate = X_all.notna().mean().sort_values(ascending=False)
full_cols = non_missing_rate[non_missing_rate >= FULL_BLOCK_MIN_RATE].index.tolist()
short_cols = non_missing_rate[non_missing_rate <= SHORT_BLOCK_MAX_RATE].index.tolist()

print("\n[BLOCK DETECT] Full block cols:", len(full_cols), "| Short block cols:", len(short_cols))

X_full = X_all[full_cols].copy() if len(full_cols) else pd.DataFrame(index=X_all.index)
X_short = X_all[short_cols].copy() if len(short_cols) else pd.DataFrame(index=X_all.index)

def block_valid(X, y):
    if X.shape[1] == 0:
        return X.copy(), y.copy(), False
    keep = X.notna().sum(axis=1) >= min(3, X.shape[1])
    X2 = X.loc[keep].copy()
    y2 = y.loc[keep].copy()
    return X2, y2, (y2.nunique() == 2)

def missing_gap_score(X, y):
    if X.shape[1] == 0:
        return np.inf
    miss_by_class = X.isna().groupby(y).mean()
    if miss_by_class.shape[0] < 2:
        return np.inf
    gap = (miss_by_class.loc[1] - miss_by_class.loc[0]).abs()
    return float(gap.mean())

Xf, yf, okf = block_valid(X_full, y)
Xs, ys, oks = block_valid(X_short, y)

gap_f = missing_gap_score(Xf, yf) if okf else np.inf
gap_s = missing_gap_score(Xs, ys) if oks else np.inf

print("\n[BLOCK SCORES]")
print("FULL  -> valid:", okf, "| rows:", len(Xf), "| gap_mean:", gap_f)
print("SHORT -> valid:", oks, "| rows:", len(Xs), "| gap_mean:", gap_s)

if gap_f <= gap_s:
    X, y = Xf, yf
    CHOSEN_BLOCK = "FULL"
else:
    X, y = Xs, ys
    CHOSEN_BLOCK = "SHORT"

print(f"\n✅ Chosen block: {CHOSEN_BLOCK}")
print("Final X shape:", X.shape)
print("Final y distribution:", Counter(y))

if y.nunique() < 2:
    raise ValueError("After block selection, only one class remains. Sheet inconsistent.")

# -------------------------------
# 5) Groups (prefer real ID if available)
# -------------------------------
def make_groups(df_raw: pd.DataFrame, X: pd.DataFrame) -> pd.Series:
    if GROUP_ID_COL is not None and GROUP_ID_COL in df_raw.columns:
        g = df_raw.loc[X.index, GROUP_ID_COL].astype(str).fillna("NA")
        return pd.Series(g.values, index=X.index)

    # Fallback: hash feature rows (duplicate-guard, not true subject grouping)
    X_for_hash = X.fillna("__MISSING__").astype(str)
    g = X_for_hash.apply(
        lambda r: hashlib.md5("||".join(r.values.tolist()).encode("utf-8")).hexdigest(),
        axis=1
    )
    return pd.Series(g.values, index=X.index)

groups = make_groups(df, X)
print("\n[GROUP CHECK] Unique groups:", groups.nunique(), "out of", len(groups))
print("[GROUP CHECK] Largest group size:", int(groups.value_counts().max()))

# -------------------------------
# 6) Splitters (outer + inner group-safe)
# -------------------------------
try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAS_SGKF = True
except Exception:
    HAS_SGKF = False
    from sklearn.model_selection import GroupKFold

def outer_splitter(seed: int):
    if HAS_SGKF:
        cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
        return cv.split(X, y, groups=groups)
    else:
        # fallback: group-only, no stratify
        gkf = GroupKFold(n_splits=N_SPLITS)
        return gkf.split(X, y, groups=groups)

def group_safe_inner_split(train_idx, val_frac, seed):
    rng = np.random.default_rng(seed)
    train_groups = groups.iloc[train_idx]
    uniq_g = train_groups.unique()
    rng.shuffle(uniq_g)
    n_val_groups = max(1, int(len(uniq_g) * val_frac))
    val_g = set(uniq_g[:n_val_groups])
    inner_val_mask = train_groups.isin(val_g).values
    inner_val_idx = train_idx[inner_val_mask]
    inner_train_idx = train_idx[~inner_val_mask]
    return inner_train_idx, inner_val_idx

def make_group_cv(seed: int, n_splits: int):
    if HAS_SGKF:
        return StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    else:
        from sklearn.model_selection import GroupKFold
        return GroupKFold(n_splits=n_splits)

# -------------------------------
# 7) Models + pipelines
# -------------------------------
def make_base_estimator(model_name, mode, spw):
    if model_name == "LR":
        cw = "balanced" if mode == "weighted" else None
        return LogisticRegression(max_iter=5000, class_weight=cw, random_state=RANDOM_STATE)

    if model_name == "SVM":
        cw = "balanced" if mode == "weighted" else None
        return SVC(kernel="rbf", probability=True, class_weight=cw, random_state=RANDOM_STATE)

    if model_name == "RF":
        cw = "balanced" if mode == "weighted" else None
        return RandomForestClassifier(n_estimators=400, class_weight=cw, random_state=RANDOM_STATE, n_jobs=-1)

    if model_name == "XGB":
        if not use_xgboost:
            return HistGradientBoostingClassifier(random_state=RANDOM_STATE)

        spw2 = spw if mode == "weighted" else 1.0
        return XGBClassifier(
            n_estimators=500,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=1.0,
            scale_pos_weight=spw2,
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

    return HistGradientBoostingClassifier(random_state=RANDOM_STATE)

def build_pipeline(model_name, mode, spw):
    imputer = ("imputer", SimpleImputer(strategy="median"))
    scaler = ("scaler", StandardScaler())
    estimator = ("model", make_base_estimator(model_name, mode, spw))

    if mode == "resampled":
        smote = ("smote", SMOTE(sampling_strategy=0.8, random_state=RANDOM_STATE))
        under = ("under", RandomUnderSampler(sampling_strategy=1.0, random_state=RANDOM_STATE))
        return ImbPipeline([imputer, scaler, smote, under, estimator])

    return SkPipeline([imputer, scaler, estimator])

def get_param_distributions(model_name):
    if model_name == "LR":
        return {"model__C": np.logspace(-3, 3, 25)}
    if model_name == "SVM":
        return {"model__C": np.logspace(-2, 3, 20), "model__gamma": ["scale", "auto"]}
    if model_name == "RF":
        return {"model__max_depth": [None, 4, 7, 12], "model__min_samples_leaf": [1, 2, 4]}
    if model_name == "XGB":
        return {"model__max_depth": [2, 3, 4], "model__learning_rate": [0.03, 0.05, 0.1]}
    return {}

# -------------------------------
# 8) Threshold selection (recall-first + precision floor)
# -------------------------------
def choose_threshold_recall_first(
    y_val, p_val,
    target_recall=TARGET_RECALL,
    min_precision=MIN_PRECISION,
    beta_fallback=BETA_FALLBACK
):
    thresholds = np.linspace(0.01, 0.99, 99)
    best = None  # (recall, precision, f1, thr)

    for t in thresholds:
        y_hat = (p_val >= t).astype(int)
        rec = recall_score(y_val, y_hat, zero_division=0)
        prec = precision_score(y_val, y_hat, zero_division=0)
        if rec >= target_recall and prec >= min_precision:
            f1 = f1_score(y_val, y_hat, zero_division=0)
            cand = (rec, prec, f1, t)
            if best is None or cand > best:
                best = cand

    if best is not None:
        rec, prec, f1, t = best
        return float(t), {"policy": "recall+precision", "val_recall": rec, "val_precision": prec, "val_f1": f1}

    # fallback: maximize F-beta
    best_t, best_score = 0.5, -1.0
    for t in thresholds:
        y_hat = (p_val >= t).astype(int)
        score = fbeta_score(y_val, y_hat, beta=beta_fallback, zero_division=0)
        if score > best_score:
            best_score = score
            best_t = t

    return float(best_t), {"policy": f"fallback_f{beta_fallback}", "val_fbeta": best_score}

# -------------------------------
# 9) Leak-safe calibration using inner-val only (Platt scaling)
# -------------------------------
def fit_platt_scaler(p_val: np.ndarray, y_val: np.ndarray):
    """
    Fit Platt scaling: logistic regression on 1D probabilities -> calibrated probabilities.
    """
    p_val = np.clip(p_val, 1e-6, 1 - 1e-6).reshape(-1, 1)
    cal = LogisticRegression(solver="lbfgs", max_iter=2000)
    cal.fit(p_val, y_val.astype(int))
    return cal

def apply_platt_scaler(cal_model, p: np.ndarray) -> np.ndarray:
    p = np.clip(p, 1e-6, 1 - 1e-6).reshape(-1, 1)
    return cal_model.predict_proba(p)[:, 1]

# -------------------------------
# 10) Shuffle sanity test (quick)
# -------------------------------
def quick_groupcv_auc(seed=42, shuffle_labels=False):
    rng = np.random.default_rng(seed)
    y_tested = y.copy()
    if shuffle_labels:
        y_tested = pd.Series(rng.permutation(y_tested.values), index=y_tested.index)

    aucs, aps = [], []
    for tr_idx, te_idx in outer_splitter(seed):
        X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
        y_tr, y_te = y_tested.iloc[tr_idx], y_tested.iloc[te_idx]
        n_pos = int((y_tr == 1).sum()); n_neg = int((y_tr == 0).sum())
        spw = (n_neg / max(n_pos, 1))

        pipe = build_pipeline("LR", "weighted", spw)
        pipe.fit(X_tr, y_tr)

        p = pipe.predict_proba(X_te)[:, 1]
        aucs.append(roc_auc_score(y_te, p))
        aps.append(average_precision_score(y_te, p))

    return float(np.mean(aucs)), float(np.mean(aps))

print("\n[SHUFFLE SANITY]")
ra, rp = quick_groupcv_auc(seed=42, shuffle_labels=False)
sa, sp = quick_groupcv_auc(seed=42, shuffle_labels=True)
print(f"Real     -> ROC-AUC={ra:.3f}, PR-AUC={rp:.3f}")
print(f"Shuffled -> ROC-AUC={sa:.3f}, PR-AUC={sp:.3f}")

# -------------------------------
# 11) Full evaluation (TRUE group-safe tuning + leak-safe calibration)
# -------------------------------
model_specs = ["LR", "SVM", "RF", "XGB"]
modes = ["weighted", "resampled"]
f2_scorer = make_scorer(fbeta_score, beta=2.0, zero_division=0)

rows = []

for rep in range(N_REPEATS):
    rep_seed = RANDOM_STATE + rep
    print(f"\n===== REPEAT {rep+1}/{N_REPEATS} | seed={rep_seed} =====")

    for model_name in model_specs:
        for mode in modes:
            print(f"\n=== {COHORT.upper()} | MODEL={model_name} | MODE={mode} ===")

            for fold, (train_idx, test_idx) in enumerate(outer_splitter(rep_seed), start=1):
                X_test = X.iloc[test_idx]
                y_test = y.iloc[test_idx]

                # inner train/val split (group-safe) for threshold + calibration
                inner_train_idx, inner_val_idx = group_safe_inner_split(
                    train_idx, INNER_VAL_FRACTION, seed=rep_seed + 1000 + fold
                )

                X_tr = X.iloc[inner_train_idx]
                y_tr = y.iloc[inner_train_idx]
                g_tr = groups.iloc[inner_train_idx]

                X_val = X.iloc[inner_val_idx]
                y_val = y.iloc[inner_val_idx]

                # scale_pos_weight (if used)
                n_pos = int((y_tr == 1).sum()); n_neg = int((y_tr == 0).sum())
                spw = (n_neg / max(n_pos, 1))

                pipe = build_pipeline(model_name, mode, spw)
                param_dist = get_param_distributions(model_name)

                # ✅ FIX 1: GROUP-SAFE inner CV for RandomizedSearchCV
                inner_cv = make_group_cv(seed=rep_seed + 2000 + fold, n_splits=INNER_SEARCH_SPLITS)

                search = RandomizedSearchCV(
                    estimator=pipe,
                    param_distributions=param_dist,
                    n_iter=SEARCH_N_ITER,
                    scoring=f2_scorer,
                    cv=inner_cv,                 # group-safe CV object
                    random_state=rep_seed,
                    n_jobs=-1,
                    refit=True
                )

                # pass groups for inner CV splitting
                if HAS_SGKF:
                    search.fit(X_tr, y_tr, groups=g_tr)
                else:
                    # GroupKFold also needs groups
                    search.fit(X_tr, y_tr, groups=g_tr)

                best_pipe = search.best_estimator_

                # Fit best model on ALL inner-train (still no leakage vs val/test)
                best_pipe.fit(X_tr, y_tr)

                # Raw probabilities on inner-val
                if hasattr(best_pipe, "predict_proba"):
                    p_val_raw = best_pipe.predict_proba(X_val)[:, 1]
                else:
                    # fallback for models without predict_proba (shouldn't happen here)
                    p_val_raw = best_pipe.decision_function(X_val)
                    p_val_raw = (p_val_raw - p_val_raw.min()) / (p_val_raw.max() - p_val_raw.min() + 1e-12)

                # ✅ FIX 2: Leak-safe calibration using ONLY inner-val (Platt scaling)
                platt = fit_platt_scaler(p_val_raw, y_val.values)

                # choose threshold on calibrated inner-val probs
                p_val_cal = apply_platt_scaler(platt, p_val_raw)
                thr, thr_info = choose_threshold_recall_first(y_val.values, p_val_cal)

                # Evaluate on outer test (calibrated)
                if hasattr(best_pipe, "predict_proba"):
                    p_test_raw = best_pipe.predict_proba(X_test)[:, 1]
                else:
                    p_test_raw = best_pipe.decision_function(X_test)
                    p_test_raw = (p_test_raw - p_test_raw.min()) / (p_test_raw.max() - p_test_raw.min() + 1e-12)

                p_test_cal = apply_platt_scaler(platt, p_test_raw)
                y_hat = (p_test_cal >= thr).astype(int)

                roc_auc = roc_auc_score(y_test, p_test_cal)
                pr_auc = average_precision_score(y_test, p_test_cal)
                brier = brier_score_loss(y_test, p_test_cal)

                prec = precision_score(y_test, y_hat, zero_division=0)
                rec = recall_score(y_test, y_hat, zero_division=0)
                f1 = f1_score(y_test, y_hat, zero_division=0)
                f2 = fbeta_score(y_test, y_hat, beta=2.0, zero_division=0)

                tn, fp, fn, tp = confusion_matrix(y_test, y_hat).ravel()
                spec = tn / (tn + fp) if (tn + fp) else 0.0

                rows.append({
                    "cohort": COHORT, "block": CHOSEN_BLOCK, "repeat": rep+1, "fold": fold,
                    "model": model_name, "mode": mode,
                    "roc_auc": roc_auc, "pr_auc": pr_auc, "brier": brier,
                    "precision": prec, "recall": rec, "specificity": spec,
                    "f1": f1, "f2": f2, "thr": thr,
                    "thr_policy": thr_info.get("policy", "unknown")
                })

                print(
                    f"Rep {rep+1} Fold {fold:02d} | thr={thr:.2f} ({thr_info.get('policy')}) | "
                    f"PR-AUC={pr_auc:.3f} ROC-AUC={roc_auc:.3f} Brier={brier:.3f} | "
                    f"R={rec:.3f} P={prec:.3f}"
                )

results_df = pd.DataFrame(rows)

summary = (
    results_df.groupby(["cohort", "block", "model", "mode"])
    .agg(
        roc_auc_mean=("roc_auc", "mean"),
        pr_auc_mean=("pr_auc", "mean"),
        brier_mean=("brier", "mean"),
        recall_mean=("recall", "mean"),
        precision_mean=("precision", "mean"),
        f2_mean=("f2", "mean"),
        thr_mean=("thr", "mean"),
        folds=("thr", "count"),
        thr_policy_mode=("thr_policy", lambda s: s.value_counts().index[0] if len(s) else "NA")
    )
    .reset_index()
    .sort_values("pr_auc_mean", ascending=False)
)

print("\n=== SUMMARY ===")
print(summary.to_string(index=False))

Loaded: antepartum shape: (1430, 22)
Sample columns: ['match_key', 'antepartum__depression', 'age', 'epds_antepartum', 'anxious_a', 'avoidant_a', 'reframing', 'selfdistraction', 'venting', 'use_of_informational', 'active', 'denial', 'religion', 'humour', 'disengagement']
Detected EPDS: epds_antepartum | cutoff: 12
y distribution: Counter({0: 1196, 1: 234})
PR-AUC baseline (prevalence): 0.164

Numeric feature pool shape: (1430, 17)
y distribution after basic align: Counter({0: 1196, 1: 234})

[BLOCK DETECT] Full block cols: 17 | Short block cols: 0

[BLOCK SCORES]
FULL  -> valid: True | rows: 1430 | gap_mean: 0.01932913633680897
SHORT -> valid: False | rows: 1430 | gap_mean: inf

✅ Chosen block: FULL
Final X shape: (1430, 17)
Final y distribution: Counter({0: 1196, 1: 234})

[GROUP CHECK] Unique groups: 1429 out of 1430
[GROUP CHECK] Largest group size: 2

[SHUFFLE SANITY]
Real     -> ROC-AUC=0.793, PR-AUC=0.492
Shuffled -> ROC-AUC=0.487, PR-AUC=0.161

===== REPEAT 1/3 | seed=42 =====



# 5

In [6]:
# 5============================================================
# ✅ FULL FIXED Kaggle Code (Block-consistent + TRUE Group-safe tuning + Leak-safe calibration)
# ------------------------------------------------------------
# Fixes (vs your current run):
# ✅ Uses a REAL group ID (auto-detected from the Excel) instead of row-hash
# ✅ Keeps your block-consistent feature selection (FULL vs SHORT)
# ✅ Inner RandomizedSearchCV is GROUP-SAFE (uses groups + StratifiedGroupKFold when available)
# ✅ Calibration is leak-safe (Platt scaling trained ONLY on inner-val)
# ✅ Threshold selected ONLY on inner-val (recall-first + precision floor)
# ✅ Adds "predicted positive rate" (alerts burden) so reviewers don't roast you
# ✅ Shuffle sanity test
# ============================================================

import warnings, hashlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from collections import Counter

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline as SkPipeline

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    precision_score, recall_score, f1_score, fbeta_score, confusion_matrix, make_scorer
)
from sklearn.model_selection import RandomizedSearchCV

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# Optional XGBoost
use_xgboost = False
try:
    from xgboost import XGBClassifier
    use_xgboost = True
except Exception:
    use_xgboost = False

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# -------------------------------
# 0) CONFIG
# -------------------------------
COHORT = "antepartum"   # "antepartum" or "postpartum"
ANTE_PATH = "/kaggle/input/datasets/maishaidaralkhateeb/ppd-depression/Antepartum_Merged_from_Postpartum.xlsx"
POST_PATH = "/kaggle/input/ppd-depression/Postpartum Depression.xlsx"

EPDS_CUTOFFS = {"antepartum": 12, "postpartum": 9}
cutoff = EPDS_CUTOFFS[COHORT]

# IMPORTANT:
# - We drop leakage proxies from FEATURES (X)
# - But we can STILL use an ID-like column for GROUPS, even if its name contains "without" etc.
LEAK_PATTERNS_X = ["outcome", "label"]  # keep this minimal for X; EPDS dropped separately

# block detection thresholds
FULL_BLOCK_MIN_RATE = 0.70
SHORT_BLOCK_MAX_RATE = 0.30

# CV settings
N_SPLITS = 5
N_REPEATS = 3
INNER_VAL_FRACTION = 0.20

# Decision policy (paper-friendly)
TARGET_RECALL = 0.85
MIN_PRECISION = 0.20
BETA_FALLBACK = 2.0

# Search
SEARCH_N_ITER = 15
INNER_SEARCH_SPLITS = 3

# -------------------------------
# 1) Load + clean headers
# -------------------------------
def load_and_clean_excel(path: str) -> pd.DataFrame:
    df = pd.read_excel(path, header=0)
    df = df.dropna(axis=1, how="all")
    df.columns = (
        df.columns.astype(str)
        .str.strip().str.lower()
        .str.replace(" ", "_")
        .str.replace(r"[^0-9a-zA-Z_]", "", regex=True)
    )
    df = df.loc[:, ~df.columns.duplicated()].copy()
    df = df.drop_duplicates()
    return df

df = load_and_clean_excel(ANTE_PATH if COHORT == "antepartum" else POST_PATH)
print("Loaded:", COHORT, "shape:", df.shape)
print("Sample columns:", df.columns.tolist()[:20])

# -------------------------------
# 2) Detect EPDS column for label
# -------------------------------
def find_best_match(cols, keywords):
    for kw in keywords:
        hits = [c for c in cols if kw in c]
        if hits:
            return hits[0]
    return None

if COHORT == "antepartum":
    epds_col = find_best_match(df.columns, ["epds_antepartum", "epdsantepartum", "epds"])
else:
    epds_col = find_best_match(df.columns, ["epds_postpartum", "epdspostpartum", "epds"])

if epds_col is None:
    raise ValueError("Could not find EPDS column.")

df[epds_col] = pd.to_numeric(df[epds_col], errors="coerce")
y = (df[epds_col] >= cutoff).astype(int)

print("Detected EPDS:", epds_col, "| cutoff:", cutoff)
print("y distribution:", Counter(y))
print("PR-AUC baseline (prevalence):", round(float(y.mean()), 3))

# -------------------------------
# 3) AUTO-DETECT a REAL group ID column (best effort)
# -------------------------------
def is_mostly_numeric(s: pd.Series, min_numeric_rate=0.95) -> bool:
    x = pd.to_numeric(s, errors="coerce")
    return float(x.notna().mean()) >= min_numeric_rate

def pick_group_id_col(df: pd.DataFrame, y: pd.Series, epds_col: str) -> str | None:
    """
    Pick a stable ID-like column to use for grouping.
    Heuristics:
      - high non-null rate
      - NOT EPDS
      - NOT mostly numeric
      - many unique values (IDs)
      - not trivially 'missing only in one class' (basic check)
    """
    n = len(df)
    candidates = []
    for c in df.columns:
        if c == epds_col:  # never
            continue
        s = df[c]
        nn_rate = float(s.notna().mean())
        if nn_rate < 0.90:
            continue
        if is_mostly_numeric(s):
            continue
        # unique rate (IDs often near 1.0, but can be lower if repeated measures)
        nunq = int(s.astype(str).fillna("NA").nunique())
        uniq_rate = nunq / max(n, 1)

        # basic "not a label-proxy by missingness":
        # if a column is nearly all missing for one class, it's suspicious
        miss_by_class = s.isna().groupby(y).mean()
        if miss_by_class.shape[0] == 2:
            miss_gap = float(abs(miss_by_class.loc[1] - miss_by_class.loc[0]))
        else:
            miss_gap = 1.0

        candidates.append((c, nn_rate, uniq_rate, miss_gap, nunq))

    if not candidates:
        return None

    cand_df = pd.DataFrame(
        candidates,
        columns=["col", "non_null_rate", "unique_rate", "miss_gap", "n_unique"]
    )

    # Prefer:
    # - non_null_rate high
    # - miss_gap low (not "only present in positives")
    # - unique_rate reasonably high
    cand_df = cand_df.sort_values(
        by=["non_null_rate", "miss_gap", "unique_rate"],
        ascending=[False, True, False]
    )

    best = cand_df.iloc[0]["col"]
    print("\n[GROUP-ID AUTO PICK] Top candidates:")
    print(cand_df.head(8).to_string(index=False))
    print("[GROUP-ID AUTO PICK] Selected:", best)
    return str(best)

GROUP_ID_COL = pick_group_id_col(df, y, epds_col)

# -------------------------------
# 4) Build numeric-only candidate feature table (X pool)
# -------------------------------
# Drop EPDS variants + obvious label-ish columns from X
drop_cols = [c for c in df.columns if any(p in c for p in LEAK_PATTERNS_X)]
drop_cols += [c for c in df.columns if "epds" in c]  # drop all epds variants

# IMPORTANT: we DO NOT drop ID col here automatically; instead we will explicitly remove it from X later.
X_all = df.drop(columns=drop_cols, errors="ignore").copy()
for c in X_all.columns:
    X_all[c] = pd.to_numeric(X_all[c], errors="coerce")
X_all = X_all.dropna(axis=1, how="all")

mask = X_all.notna().any(axis=1) & y.notna()
X_all = X_all.loc[mask].copy()
y = y.loc[mask].copy()
df_aligned = df.loc[mask].copy()  # keep aligned raw df for groups

print("\nNumeric feature pool shape:", X_all.shape)
print("y distribution after align:", Counter(y))

# -------------------------------
# 5) Identify blocks by coverage (FULL vs SHORT) and choose ONE
# -------------------------------
non_missing_rate = X_all.notna().mean().sort_values(ascending=False)
full_cols = non_missing_rate[non_missing_rate >= FULL_BLOCK_MIN_RATE].index.tolist()
short_cols = non_missing_rate[non_missing_rate <= SHORT_BLOCK_MAX_RATE].index.tolist()

print("\n[BLOCK DETECT] Full block cols:", len(full_cols), "| Short block cols:", len(short_cols))

X_full = X_all[full_cols].copy() if len(full_cols) else pd.DataFrame(index=X_all.index)
X_short = X_all[short_cols].copy() if len(short_cols) else pd.DataFrame(index=X_all.index)

def block_valid(X, y):
    if X.shape[1] == 0:
        return X.copy(), y.copy(), False
    keep = X.notna().sum(axis=1) >= min(3, X.shape[1])
    X2 = X.loc[keep].copy()
    y2 = y.loc[keep].copy()
    return X2, y2, (y2.nunique() == 2)

def missing_gap_score(X, y):
    if X.shape[1] == 0:
        return np.inf
    miss_by_class = X.isna().groupby(y).mean()
    if miss_by_class.shape[0] < 2:
        return np.inf
    gap = (miss_by_class.loc[1] - miss_by_class.loc[0]).abs()
    return float(gap.mean())

Xf, yf, okf = block_valid(X_full, y)
Xs, ys, oks = block_valid(X_short, y)

gap_f = missing_gap_score(Xf, yf) if okf else np.inf
gap_s = missing_gap_score(Xs, ys) if oks else np.inf

print("\n[BLOCK SCORES]")
print("FULL  -> valid:", okf, "| rows:", len(Xf), "| gap_mean:", gap_f)
print("SHORT -> valid:", oks, "| rows:", len(Xs), "| gap_mean:", gap_s)

if gap_f <= gap_s:
    X, y = Xf, yf
    CHOSEN_BLOCK = "FULL"
else:
    X, y = Xs, ys
    CHOSEN_BLOCK = "SHORT"

# Explicitly remove GROUP_ID_COL from X if it exists there (never model on IDs)
if GROUP_ID_COL is not None and GROUP_ID_COL in X.columns:
    X = X.drop(columns=[GROUP_ID_COL])

print(f"\n✅ Chosen block: {CHOSEN_BLOCK}")
print("Final X shape:", X.shape)
print("Final y distribution:", Counter(y))

if y.nunique() < 2:
    raise ValueError("After block selection, only one class remains.")

# -------------------------------
# 6) Build groups (REAL ID if available, else fallback to row-hash)
# -------------------------------
def make_groups(df_raw_aligned: pd.DataFrame, X: pd.DataFrame, group_col: str | None) -> pd.Series:
    if group_col is not None and group_col in df_raw_aligned.columns:
        g = df_raw_aligned.loc[X.index, group_col].astype(str).fillna("NA")
        return pd.Series(g.values, index=X.index)

    # fallback: row-hash
    X_for_hash = X.fillna("__MISSING__").astype(str)
    g = X_for_hash.apply(lambda r: hashlib.md5("||".join(r.values.tolist()).encode("utf-8")).hexdigest(), axis=1)
    return pd.Series(g.values, index=X.index)

groups = make_groups(df_aligned, X, GROUP_ID_COL)

print("\n[GROUP CHECK] Using GROUP_ID_COL =", GROUP_ID_COL)
print("[GROUP CHECK] Unique groups:", groups.nunique(), "out of", len(groups))
print("[GROUP CHECK] Largest group size:", int(groups.value_counts().max()))
print("[GROUP CHECK] % NA groups:", round(float((groups == "NA").mean()), 4))

# -------------------------------
# 7) Splitters (outer + inner group-safe)
# -------------------------------
try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAS_SGKF = True
except Exception:
    HAS_SGKF = False
    from sklearn.model_selection import GroupKFold

def outer_splitter(seed: int):
    if HAS_SGKF:
        cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
        return cv.split(X, y, groups=groups)
    else:
        gkf = GroupKFold(n_splits=N_SPLITS)
        return gkf.split(X, y, groups=groups)

def group_safe_inner_split(train_idx, val_frac, seed):
    rng = np.random.default_rng(seed)
    train_groups = groups.iloc[train_idx]
    uniq_g = train_groups.unique()
    rng.shuffle(uniq_g)
    n_val_groups = max(1, int(len(uniq_g) * val_frac))
    val_g = set(uniq_g[:n_val_groups])
    inner_val_mask = train_groups.isin(val_g).values
    inner_val_idx = train_idx[inner_val_mask]
    inner_train_idx = train_idx[~inner_val_mask]
    return inner_train_idx, inner_val_idx

def make_group_cv(seed: int, n_splits: int):
    if HAS_SGKF:
        return StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    else:
        from sklearn.model_selection import GroupKFold
        return GroupKFold(n_splits=n_splits)

# -------------------------------
# 8) Models + pipelines
# -------------------------------
def make_base_estimator(model_name, mode, spw):
    if model_name == "LR":
        cw = "balanced" if mode == "weighted" else None
        return LogisticRegression(max_iter=5000, class_weight=cw, random_state=RANDOM_STATE)

    if model_name == "SVM":
        cw = "balanced" if mode == "weighted" else None
        return SVC(kernel="rbf", probability=True, class_weight=cw, random_state=RANDOM_STATE)

    if model_name == "RF":
        cw = "balanced" if mode == "weighted" else None
        return RandomForestClassifier(n_estimators=400, class_weight=cw, random_state=RANDOM_STATE, n_jobs=-1)

    if model_name == "XGB":
        if not use_xgboost:
            return HistGradientBoostingClassifier(random_state=RANDOM_STATE)

        spw2 = spw if mode == "weighted" else 1.0
        return XGBClassifier(
            n_estimators=500, max_depth=3, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
            scale_pos_weight=spw2, eval_metric="logloss",
            random_state=RANDOM_STATE, n_jobs=-1
        )

    return HistGradientBoostingClassifier(random_state=RANDOM_STATE)

def build_pipeline(model_name, mode, spw):
    imputer = ("imputer", SimpleImputer(strategy="median"))
    scaler = ("scaler", StandardScaler())
    estimator = ("model", make_base_estimator(model_name, mode, spw))

    if mode == "resampled":
        smote = ("smote", SMOTE(sampling_strategy=0.8, random_state=RANDOM_STATE))
        under = ("under", RandomUnderSampler(sampling_strategy=1.0, random_state=RANDOM_STATE))
        return ImbPipeline([imputer, scaler, smote, under, estimator])

    return SkPipeline([imputer, scaler, estimator])

def get_param_distributions(model_name):
    if model_name == "LR":
        return {"model__C": np.logspace(-3, 3, 25)}
    if model_name == "SVM":
        return {"model__C": np.logspace(-2, 3, 20), "model__gamma": ["scale", "auto"]}
    if model_name == "RF":
        return {"model__max_depth": [None, 4, 7, 12], "model__min_samples_leaf": [1, 2, 4]}
    if model_name == "XGB":
        return {"model__max_depth": [2, 3, 4], "model__learning_rate": [0.03, 0.05, 0.1]}
    return {}

# -------------------------------
# 9) Threshold selection (recall-first + precision floor)
# -------------------------------
def choose_threshold_recall_first(y_val, p_val, target_recall=TARGET_RECALL, min_precision=MIN_PRECISION, beta_fallback=BETA_FALLBACK):
    thresholds = np.linspace(0.01, 0.99, 99)
    best = None  # (recall, precision, f1, thr)

    for t in thresholds:
        y_hat = (p_val >= t).astype(int)
        rec = recall_score(y_val, y_hat, zero_division=0)
        prec = precision_score(y_val, y_hat, zero_division=0)
        if rec >= target_recall and prec >= min_precision:
            f1 = f1_score(y_val, y_hat, zero_division=0)
            cand = (rec, prec, f1, t)
            if best is None or cand > best:
                best = cand

    if best is not None:
        rec, prec, f1, t = best
        return float(t), {"policy": "recall+precision", "val_recall": rec, "val_precision": prec, "val_f1": f1}

    # fallback: maximize F-beta
    best_t, best_score = 0.5, -1.0
    for t in thresholds:
        y_hat = (p_val >= t).astype(int)
        score = fbeta_score(y_val, y_hat, beta=beta_fallback, zero_division=0)
        if score > best_score:
            best_score = score
            best_t = t

    return float(best_t), {"policy": f"fallback_f{beta_fallback}", "val_fbeta": best_score}

# -------------------------------
# 10) Leak-safe calibration using inner-val only (Platt scaling)
# -------------------------------
def fit_platt_scaler(p_val: np.ndarray, y_val: np.ndarray):
    p_val = np.clip(p_val, 1e-6, 1 - 1e-6).reshape(-1, 1)
    cal = LogisticRegression(solver="lbfgs", max_iter=2000)
    cal.fit(p_val, y_val.astype(int))
    return cal

def apply_platt_scaler(cal_model, p: np.ndarray) -> np.ndarray:
    p = np.clip(p, 1e-6, 1 - 1e-6).reshape(-1, 1)
    return cal_model.predict_proba(p)[:, 1]

# -------------------------------
# 11) Shuffle sanity test (quick)
# -------------------------------
def quick_groupcv_auc(seed=42, shuffle_labels=False):
    rng = np.random.default_rng(seed)
    y_tested = y.copy()
    if shuffle_labels:
        y_tested = pd.Series(rng.permutation(y_tested.values), index=y_tested.index)

    aucs, aps = [], []
    for tr_idx, te_idx in outer_splitter(seed):
        X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
        y_tr, y_te = y_tested.iloc[tr_idx], y_tested.iloc[te_idx]

        n_pos = int((y_tr == 1).sum()); n_neg = int((y_tr == 0).sum())
        spw = (n_neg / max(n_pos, 1))

        pipe = build_pipeline("LR", "weighted", spw)
        pipe.fit(X_tr, y_tr)
        p = pipe.predict_proba(X_te)[:, 1]

        aucs.append(roc_auc_score(y_te, p))
        aps.append(average_precision_score(y_te, p))

    return float(np.mean(aucs)), float(np.mean(aps))

print("\n[SHUFFLE SANITY]")
ra, rp = quick_groupcv_auc(seed=42, shuffle_labels=False)
sa, sp = quick_groupcv_auc(seed=42, shuffle_labels=True)
print(f"Real     -> ROC-AUC={ra:.3f}, PR-AUC={rp:.3f}")
print(f"Shuffled -> ROC-AUC={sa:.3f}, PR-AUC={sp:.3f}")

# -------------------------------
# 12) Full evaluation
# -------------------------------
model_specs = ["LR", "SVM", "RF", "XGB"]
modes = ["weighted", "resampled"]
f2_scorer = make_scorer(fbeta_score, beta=2.0, zero_division=0)

rows = []

for rep in range(N_REPEATS):
    rep_seed = RANDOM_STATE + rep
    print(f"\n===== REPEAT {rep+1}/{N_REPEATS} | seed={rep_seed} =====")

    for model_name in model_specs:
        for mode in modes:
            print(f"\n=== {COHORT.upper()} | MODEL={model_name} | MODE={mode} ===")

            for fold, (train_idx, test_idx) in enumerate(outer_splitter(rep_seed), start=1):
                X_test = X.iloc[test_idx]
                y_test = y.iloc[test_idx]

                inner_train_idx, inner_val_idx = group_safe_inner_split(
                    train_idx, INNER_VAL_FRACTION, seed=rep_seed + 1000 + fold
                )

                X_tr = X.iloc[inner_train_idx]
                y_tr = y.iloc[inner_train_idx]
                g_tr = groups.iloc[inner_train_idx]

                X_val = X.iloc[inner_val_idx]
                y_val = y.iloc[inner_val_idx]

                n_pos = int((y_tr == 1).sum()); n_neg = int((y_tr == 0).sum())
                spw = (n_neg / max(n_pos, 1))

                pipe = build_pipeline(model_name, mode, spw)
                param_dist = get_param_distributions(model_name)

                inner_cv = make_group_cv(seed=rep_seed + 2000 + fold, n_splits=INNER_SEARCH_SPLITS)

                search = RandomizedSearchCV(
                    estimator=pipe,
                    param_distributions=param_dist,
                    n_iter=SEARCH_N_ITER,
                    scoring=f2_scorer,
                    cv=inner_cv,
                    random_state=rep_seed,
                    n_jobs=-1,
                    refit=True
                )

                search.fit(X_tr, y_tr, groups=g_tr)
                best_pipe = search.best_estimator_
                best_pipe.fit(X_tr, y_tr)

                # raw probs on val
                p_val_raw = best_pipe.predict_proba(X_val)[:, 1]
                platt = fit_platt_scaler(p_val_raw, y_val.values)

                p_val_cal = apply_platt_scaler(platt, p_val_raw)
                thr, thr_info = choose_threshold_recall_first(y_val.values, p_val_cal)

                # calibrated probs on test
                p_test_raw = best_pipe.predict_proba(X_test)[:, 1]
                p_test_cal = apply_platt_scaler(platt, p_test_raw)

                y_hat = (p_test_cal >= thr).astype(int)

                roc_auc = roc_auc_score(y_test, p_test_cal)
                pr_auc = average_precision_score(y_test, p_test_cal)
                brier = brier_score_loss(y_test, p_test_cal)

                prec = precision_score(y_test, y_hat, zero_division=0)
                rec = recall_score(y_test, y_hat, zero_division=0)
                f1 = f1_score(y_test, y_hat, zero_division=0)
                f2 = fbeta_score(y_test, y_hat, beta=2.0, zero_division=0)

                tn, fp, fn, tp = confusion_matrix(y_test, y_hat).ravel()
                spec = tn / (tn + fp) if (tn + fp) else 0.0
                pred_pos_rate = float(np.mean(y_hat))

                rows.append({
                    "cohort": COHORT, "block": CHOSEN_BLOCK, "repeat": rep+1, "fold": fold,
                    "model": model_name, "mode": mode,
                    "roc_auc": roc_auc, "pr_auc": pr_auc, "brier": brier,
                    "precision": prec, "recall": rec, "specificity": spec,
                    "f1": f1, "f2": f2, "thr": thr,
                    "thr_policy": thr_info.get("policy", "unknown"),
                    "pred_pos_rate": pred_pos_rate
                })

                print(
                    f"Rep {rep+1} Fold {fold:02d} | thr={thr:.2f} ({thr_info.get('policy')}) | "
                    f"PR-AUC={pr_auc:.3f} ROC-AUC={roc_auc:.3f} Brier={brier:.3f} | "
                    f"R={rec:.3f} P={prec:.3f} | Pred+={pred_pos_rate:.3f}"
                )

results_df = pd.DataFrame(rows)

summary = (
    results_df.groupby(["cohort", "block", "model", "mode"])
    .agg(
        roc_auc_mean=("roc_auc", "mean"),
        pr_auc_mean=("pr_auc", "mean"),
        brier_mean=("brier", "mean"),
        recall_mean=("recall", "mean"),
        precision_mean=("precision", "mean"),
        f2_mean=("f2", "mean"),
        thr_mean=("thr", "mean"),
        pred_pos_rate_mean=("pred_pos_rate", "mean"),
        folds=("thr", "count"),
        thr_policy_mode=("thr_policy", lambda s: s.value_counts().index[0] if len(s) else "NA")
    )
    .reset_index()
    .sort_values("pr_auc_mean", ascending=False)
)

print("\n=== SUMMARY ===")
print(summary.to_string(index=False))

Loaded: antepartum shape: (1430, 22)
Sample columns: ['match_key', 'antepartum__depression', 'age', 'epds_antepartum', 'anxious_a', 'avoidant_a', 'reframing', 'selfdistraction', 'venting', 'use_of_informational', 'active', 'denial', 'religion', 'humour', 'disengagement', 'emotional_s', 'substance', 'acceptance', 'planning', 'selfblame']
Detected EPDS: epds_antepartum | cutoff: 12
y distribution: Counter({0: 1196, 1: 234})
PR-AUC baseline (prevalence): 0.164

[GROUP-ID AUTO PICK] Top candidates:
                                 col  non_null_rate  unique_rate  miss_gap  n_unique
women_without_antepartum__depression       1.000000     1.000000   0.00000      1430
                           match_key       1.000000     0.997203   0.00000      1426
                                 age       0.931469     0.025175   0.08194        36
                     selfdistraction       0.931469     0.005594   0.08194         8
                         emotional_s       0.931469     0.005594   0.08194 

In [7]:
# 6============================================================
# ✅ FULL FIXED Kaggle Code
# Block-consistent + TRUE Group-safe tuning + Leak-safe calibration
# + Robust group-id picking + Burden-capped thresholding + Refit-on-full-train
# ============================================================

import warnings, hashlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from collections import Counter

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline as SkPipeline

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    precision_score, recall_score, f1_score, fbeta_score, confusion_matrix, make_scorer
)
from sklearn.model_selection import RandomizedSearchCV

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# Optional XGBoost
use_xgboost = False
try:
    from xgboost import XGBClassifier
    use_xgboost = True
except Exception:
    use_xgboost = False

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# -------------------------------
# 0) CONFIG
# -------------------------------
COHORT = "antepartum"   # "antepartum" or "postpartum"
ANTE_PATH = "/kaggle/input/datasets/maishaidaralkhateeb/ppd-depression/Antepartum_Merged_from_Postpartum.xlsx"
POST_PATH = "/kaggle/input/ppd-depression/Postpartum Depression.xlsx"

EPDS_CUTOFFS = {"antepartum": 12, "postpartum": 9}
cutoff = EPDS_CUTOFFS[COHORT]

# Drop leakage proxies from FEATURES (X); EPDS dropped separately
LEAK_PATTERNS_X = ["outcome", "label"]

# GROUP ID safety: ban suspicious group candidates
GROUP_LEAK_TOKENS = [
    "epds", "depress", "ppd", "outcome", "label",
    "without", "case", "control", "status", "screen"
]

# block detection thresholds
FULL_BLOCK_MIN_RATE = 0.70
SHORT_BLOCK_MAX_RATE = 0.30

# CV settings
N_SPLITS = 5
N_REPEATS = 3
INNER_VAL_FRACTION = 0.20

# Decision policy
TARGET_RECALL = 0.85
MIN_PRECISION = 0.20
BETA_FALLBACK = 2.0

# Search
SEARCH_N_ITER = 15
INNER_SEARCH_SPLITS = 3

# -------------------------------
# 1) Load + clean headers
# -------------------------------
def load_and_clean_excel(path: str) -> pd.DataFrame:
    df = pd.read_excel(path, header=0)
    df = df.dropna(axis=1, how="all")
    df.columns = (
        df.columns.astype(str)
        .str.strip().str.lower()
        .str.replace(" ", "_")
        .str.replace(r"[^0-9a-zA-Z_]", "", regex=True)
    )
    df = df.loc[:, ~df.columns.duplicated()].copy()
    df = df.drop_duplicates()
    return df

df = load_and_clean_excel(ANTE_PATH if COHORT == "antepartum" else POST_PATH)
print("Loaded:", COHORT, "shape:", df.shape)
print("Sample columns:", df.columns.tolist()[:25])

# -------------------------------
# 2) Detect EPDS column for label
# -------------------------------
def find_best_match(cols, keywords):
    for kw in keywords:
        hits = [c for c in cols if kw in c]
        if hits:
            return hits[0]
    return None

if COHORT == "antepartum":
    epds_col = find_best_match(df.columns, ["epds_antepartum", "epdsantepartum", "epds"])
else:
    epds_col = find_best_match(df.columns, ["epds_postpartum", "epdspostpartum", "epds"])

if epds_col is None:
    raise ValueError("Could not find EPDS column.")

df[epds_col] = pd.to_numeric(df[epds_col], errors="coerce")
y = (df[epds_col] >= cutoff).astype(int)

prev = float(y.mean())
print("Detected EPDS:", epds_col, "| cutoff:", cutoff)
print("y distribution:", Counter(y))
print("Positive prevalence:", round(prev, 3), "| AP baseline≈prevalence")

# -------------------------------
# 3) AUTO-DETECT a REAL group ID column (best effort, safer)
# -------------------------------
def is_mostly_numeric(s: pd.Series, min_numeric_rate=0.95) -> bool:
    x = pd.to_numeric(s, errors="coerce")
    return float(x.notna().mean()) >= min_numeric_rate

def looks_like_id(colname: str) -> bool:
    return any(k in colname for k in ["id", "pid", "participant", "subject", "woman", "mother", "record"])

def pick_group_id_col(df: pd.DataFrame, y: pd.Series, epds_col: str) -> str | None:
    n = len(df)
    candidates = []

    for c in df.columns:
        if c == epds_col:
            continue

        # hard-ban suspicious group candidates
        if any(tok in c for tok in GROUP_LEAK_TOKENS):
            continue

        s = df[c]
        nn_rate = float(s.notna().mean())
        if nn_rate < 0.90:
            continue

        # allow numeric IDs IF column name looks like an ID
        if is_mostly_numeric(s) and not looks_like_id(c):
            continue

        nunq = int(s.astype(str).fillna("NA").nunique())
        uniq_rate = nunq / max(n, 1)

        miss_by_class = s.isna().groupby(y).mean()
        if miss_by_class.shape[0] == 2:
            miss_gap = float(abs(miss_by_class.loc[1] - miss_by_class.loc[0]))
        else:
            miss_gap = 1.0

        # score includes "looks_like_id" boost
        id_boost = 1.0 if looks_like_id(c) else 0.0
        candidates.append((c, nn_rate, uniq_rate, miss_gap, nunq, id_boost))

    if not candidates:
        return None

    cand_df = pd.DataFrame(
        candidates,
        columns=["col", "non_null_rate", "unique_rate", "miss_gap", "n_unique", "id_boost"]
    )

    cand_df = cand_df.sort_values(
        by=["id_boost", "non_null_rate", "miss_gap", "unique_rate"],
        ascending=[False, False, True, False]
    )

    best = str(cand_df.iloc[0]["col"])
    print("\n[GROUP-ID AUTO PICK] Top candidates:")
    print(cand_df.head(8).to_string(index=False))
    print("[GROUP-ID AUTO PICK] Selected:", best)
    return best

GROUP_ID_COL = pick_group_id_col(df, y, epds_col)

# -------------------------------
# 4) Build numeric-only candidate feature table (X pool)
# -------------------------------
drop_cols = [c for c in df.columns if any(p in c for p in LEAK_PATTERNS_X)]
drop_cols += [c for c in df.columns if "epds" in c]  # drop all epds variants

X_all = df.drop(columns=drop_cols, errors="ignore").copy()
for c in X_all.columns:
    X_all[c] = pd.to_numeric(X_all[c], errors="coerce")
X_all = X_all.dropna(axis=1, how="all")

mask = X_all.notna().any(axis=1) & y.notna()
X_all = X_all.loc[mask].copy()
y = y.loc[mask].copy()
df_aligned = df.loc[mask].copy()

print("\nNumeric feature pool shape:", X_all.shape)
print("y distribution after align:", Counter(y))

# -------------------------------
# 5) Identify blocks (FULL vs SHORT), choose ONE
# -------------------------------
non_missing_rate = X_all.notna().mean().sort_values(ascending=False)
full_cols = non_missing_rate[non_missing_rate >= FULL_BLOCK_MIN_RATE].index.tolist()
short_cols = non_missing_rate[non_missing_rate <= SHORT_BLOCK_MAX_RATE].index.tolist()

print("\n[BLOCK DETECT] Full block cols:", len(full_cols), "| Short block cols:", len(short_cols))

X_full = X_all[full_cols].copy() if len(full_cols) else pd.DataFrame(index=X_all.index)
X_short = X_all[short_cols].copy() if len(short_cols) else pd.DataFrame(index=X_all.index)

def block_valid(Xb, yb):
    if Xb.shape[1] == 0:
        return Xb.copy(), yb.copy(), False
    keep = Xb.notna().sum(axis=1) >= min(3, Xb.shape[1])
    X2 = Xb.loc[keep].copy()
    y2 = yb.loc[keep].copy()
    return X2, y2, (y2.nunique() == 2)

def missing_gap_score(Xb, yb):
    if Xb.shape[1] == 0:
        return np.inf
    miss_by_class = Xb.isna().groupby(yb).mean()
    if miss_by_class.shape[0] < 2:
        return np.inf
    gap = (miss_by_class.loc[1] - miss_by_class.loc[0]).abs()
    return float(gap.mean())

Xf, yf, okf = block_valid(X_full, y)
Xs, ys, oks = block_valid(X_short, y)

gap_f = missing_gap_score(Xf, yf) if okf else np.inf
gap_s = missing_gap_score(Xs, ys) if oks else np.inf

print("\n[BLOCK SCORES]")
print("FULL  -> valid:", okf, "| rows:", len(Xf), "| gap_mean:", gap_f)
print("SHORT -> valid:", oks, "| rows:", len(Xs), "| gap_mean:", gap_s)

if gap_f <= gap_s:
    X, y = Xf, yf
    CHOSEN_BLOCK = "FULL"
else:
    X, y = Xs, ys
    CHOSEN_BLOCK = "SHORT"

# -------------------------------
# 6) Build groups (REAL ID if available, else fallback to row-hash)
# -------------------------------
def make_groups(df_raw_aligned: pd.DataFrame, X: pd.DataFrame, group_col: str | None) -> pd.Series:
    if group_col is not None and group_col in df_raw_aligned.columns:
        g = df_raw_aligned.loc[X.index, group_col].astype(str).fillna("NA")
        return pd.Series(g.values, index=X.index)

    X_for_hash = X.fillna("__MISSING__").astype(str)
    g = X_for_hash.apply(lambda r: hashlib.md5("||".join(r.values.tolist()).encode("utf-8")).hexdigest(), axis=1)
    return pd.Series(g.values, index=X.index)

groups = make_groups(df_aligned, X, GROUP_ID_COL)

# Never model on group IDs
if GROUP_ID_COL is not None and GROUP_ID_COL in X.columns:
    X = X.drop(columns=[GROUP_ID_COL])

print(f"\n✅ Chosen block: {CHOSEN_BLOCK}")
print("Final X shape:", X.shape)
print("Final y distribution:", Counter(y))

print("\n[GROUP CHECK] Using GROUP_ID_COL =", GROUP_ID_COL)
print("[GROUP CHECK] Unique groups:", groups.nunique(), "out of", len(groups))
print("[GROUP CHECK] Largest group size:", int(groups.value_counts().max()))
print("[GROUP CHECK] % NA groups:", round(float((groups == "NA").mean()), 4))

if y.nunique() < 2:
    raise ValueError("After block selection, only one class remains.")

# Burden cap: no more than 2x prevalence (and never above 0.40)
MAX_PRED_POS_RATE = min(2.0 * float(y.mean()), 0.40)
print("\n[THRESHOLD BURDEN CAP] MAX_PRED_POS_RATE =", round(MAX_PRED_POS_RATE, 3))

# -------------------------------
# 7) Splitters (outer + inner group-safe)
# -------------------------------
try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAS_SGKF = True
except Exception:
    HAS_SGKF = False
    from sklearn.model_selection import GroupKFold

def outer_splitter(seed: int):
    if HAS_SGKF:
        cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
        return cv.split(X, y, groups=groups)
    else:
        from sklearn.model_selection import GroupKFold
        gkf = GroupKFold(n_splits=N_SPLITS)
        return gkf.split(X, y, groups=groups)

def group_safe_inner_split(train_idx, val_frac, seed):
    rng = np.random.default_rng(seed)
    train_groups = groups.iloc[train_idx]
    uniq_g = train_groups.unique()
    rng.shuffle(uniq_g)
    n_val_groups = max(1, int(len(uniq_g) * val_frac))
    val_g = set(uniq_g[:n_val_groups])
    inner_val_mask = train_groups.isin(val_g).values
    inner_val_idx = train_idx[inner_val_mask]
    inner_train_idx = train_idx[~inner_val_mask]
    return inner_train_idx, inner_val_idx

def make_group_cv(seed: int, n_splits: int):
    if HAS_SGKF:
        return StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    else:
        from sklearn.model_selection import GroupKFold
        return GroupKFold(n_splits=n_splits)

# -------------------------------
# 8) Models + pipelines
# -------------------------------
def make_base_estimator(model_name, mode, spw):
    if model_name == "LR":
        cw = "balanced" if mode == "weighted" else None
        return LogisticRegression(max_iter=5000, class_weight=cw, random_state=RANDOM_STATE)

    if model_name == "SVM":
        cw = "balanced" if mode == "weighted" else None
        return SVC(kernel="rbf", probability=True, class_weight=cw, random_state=RANDOM_STATE)

    if model_name == "RF":
        cw = "balanced" if mode == "weighted" else None
        return RandomForestClassifier(n_estimators=400, class_weight=cw, random_state=RANDOM_STATE, n_jobs=-1)

    if model_name == "XGB":
        if not use_xgboost:
            return HistGradientBoostingClassifier(random_state=RANDOM_STATE)

        spw2 = spw if mode == "weighted" else 1.0
        return XGBClassifier(
            n_estimators=500, max_depth=3, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
            scale_pos_weight=spw2, eval_metric="logloss",
            random_state=RANDOM_STATE, n_jobs=-1
        )

    return HistGradientBoostingClassifier(random_state=RANDOM_STATE)

def build_pipeline(model_name, mode, spw):
    imputer = ("imputer", SimpleImputer(strategy="median"))
    scaler = ("scaler", StandardScaler())
    estimator = ("model", make_base_estimator(model_name, mode, spw))

    if mode == "resampled":
        smote = ("smote", SMOTE(sampling_strategy=0.8, random_state=RANDOM_STATE))
        under = ("under", RandomUnderSampler(sampling_strategy=1.0, random_state=RANDOM_STATE))
        return ImbPipeline([imputer, scaler, smote, under, estimator])

    return SkPipeline([imputer, scaler, estimator])

def get_param_distributions(model_name):
    if model_name == "LR":
        return {"model__C": np.logspace(-3, 3, 25)}
    if model_name == "SVM":
        return {"model__C": np.logspace(-2, 3, 20), "model__gamma": ["scale", "auto"]}
    if model_name == "RF":
        return {"model__max_depth": [None, 4, 7, 12], "model__min_samples_leaf": [1, 2, 4]}
    if model_name == "XGB":
        return {"model__max_depth": [2, 3, 4], "model__learning_rate": [0.03, 0.05, 0.1]}
    return {}

# -------------------------------
# 9) Threshold selection (recall-first + precision floor + burden cap)
# -------------------------------
def choose_threshold_recall_first(y_val, p_val,
                                 target_recall=TARGET_RECALL,
                                 min_precision=MIN_PRECISION,
                                 beta_fallback=BETA_FALLBACK,
                                 max_pred_pos_rate=None):
    thresholds = np.linspace(0.01, 0.99, 99)
    best = None  # (recall, precision, f1, -pred_pos, thr)

    for t in thresholds:
        y_hat = (p_val >= t).astype(int)
        pred_pos = float(np.mean(y_hat))
        if max_pred_pos_rate is not None and pred_pos > max_pred_pos_rate:
            continue

        rec = recall_score(y_val, y_hat, zero_division=0)
        prec = precision_score(y_val, y_hat, zero_division=0)
        if rec >= target_recall and prec >= min_precision:
            f1 = f1_score(y_val, y_hat, zero_division=0)
            cand = (rec, prec, f1, -pred_pos, t)
            if best is None or cand > best:
                best = cand

    if best is not None:
        rec, prec, f1, _, t = best
        return float(t), {"policy": "recall+precision+burden", "val_recall": rec, "val_precision": prec, "val_f1": f1}

    # fallback: maximize F-beta under the SAME burden cap
    best_t, best_score = 0.5, -1.0
    for t in thresholds:
        y_hat = (p_val >= t).astype(int)
        pred_pos = float(np.mean(y_hat))
        if max_pred_pos_rate is not None and pred_pos > max_pred_pos_rate:
            continue
        score = fbeta_score(y_val, y_hat, beta=beta_fallback, zero_division=0)
        if score > best_score:
            best_score = score
            best_t = t

    return float(best_t), {"policy": f"fallback_f{beta_fallback}_burden", "val_fbeta": best_score}

# -------------------------------
# 10) Leak-safe calibration (Platt scaling on inner-val only)
# -------------------------------
def fit_platt_scaler(p_val: np.ndarray, y_val: np.ndarray):
    p_val = np.clip(p_val, 1e-6, 1 - 1e-6).reshape(-1, 1)
    cal = LogisticRegression(solver="lbfgs", max_iter=2000)
    cal.fit(p_val, y_val.astype(int))
    return cal

def apply_platt_scaler(cal_model, p: np.ndarray) -> np.ndarray:
    p = np.clip(p, 1e-6, 1 - 1e-6).reshape(-1, 1)
    return cal_model.predict_proba(p)[:, 1]

# -------------------------------
# 11) Shuffle sanity test (quick)
# -------------------------------
def quick_groupcv_auc(seed=42, shuffle_labels=False):
    rng = np.random.default_rng(seed)
    y_tested = y.copy()
    if shuffle_labels:
        y_tested = pd.Series(rng.permutation(y_tested.values), index=y_tested.index)

    aucs, aps = [], []
    for tr_idx, te_idx in outer_splitter(seed):
        X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
        y_tr, y_te = y_tested.iloc[tr_idx], y_tested.iloc[te_idx]

        n_pos = int((y_tr == 1).sum()); n_neg = int((y_tr == 0).sum())
        spw = (n_neg / max(n_pos, 1))

        pipe = build_pipeline("LR", "weighted", spw)
        pipe.fit(X_tr, y_tr)
        p = pipe.predict_proba(X_te)[:, 1]

        aucs.append(roc_auc_score(y_te, p))
        aps.append(average_precision_score(y_te, p))

    return float(np.mean(aucs)), float(np.mean(aps))

print("\n[SHUFFLE SANITY]")
ra, rp = quick_groupcv_auc(seed=42, shuffle_labels=False)
sa, sp = quick_groupcv_auc(seed=42, shuffle_labels=True)
print(f"Real     -> ROC-AUC={ra:.3f}, PR-AUC={rp:.3f}")
print(f"Shuffled -> ROC-AUC={sa:.3f}, PR-AUC={sp:.3f}")

# -------------------------------
# 12) Full evaluation (cross-fit style + refit on full outer-train)
# -------------------------------
model_specs = ["LR", "SVM", "RF", "XGB"]
modes = ["weighted", "resampled"]
f2_scorer = make_scorer(fbeta_score, beta=2.0, zero_division=0)

rows = []

for rep in range(N_REPEATS):
    rep_seed = RANDOM_STATE + rep
    print(f"\n===== REPEAT {rep+1}/{N_REPEATS} | seed={rep_seed} =====")

    for model_name in model_specs:
        for mode in modes:
            print(f"\n=== {COHORT.upper()} | MODEL={model_name} | MODE={mode} ===")

            for fold, (train_idx, test_idx) in enumerate(outer_splitter(rep_seed), start=1):
                X_test = X.iloc[test_idx]
                y_test = y.iloc[test_idx]

                inner_train_idx, inner_val_idx = group_safe_inner_split(
                    train_idx, INNER_VAL_FRACTION, seed=rep_seed + 1000 + fold
                )

                X_tr = X.iloc[inner_train_idx]
                y_tr = y.iloc[inner_train_idx]
                g_tr = groups.iloc[inner_train_idx]

                X_val = X.iloc[inner_val_idx]
                y_val = y.iloc[inner_val_idx]

                n_pos = int((y_tr == 1).sum()); n_neg = int((y_tr == 0).sum())
                spw = (n_neg / max(n_pos, 1))

                # ----- group-safe hyperparameter search on inner_train only -----
                pipe = build_pipeline(model_name, mode, spw)
                param_dist = get_param_distributions(model_name)
                inner_cv = make_group_cv(seed=rep_seed + 2000 + fold, n_splits=INNER_SEARCH_SPLITS)

                search = RandomizedSearchCV(
                    estimator=pipe,
                    param_distributions=param_dist,
                    n_iter=SEARCH_N_ITER,
                    scoring=f2_scorer,
                    cv=inner_cv,
                    random_state=rep_seed,
                    n_jobs=-1,
                    refit=True
                )
                search.fit(X_tr, y_tr, groups=g_tr)

                best_params = search.best_params_

                # ----- (1) Fit on inner_train -> calibrate + threshold using inner_val -----
                best_pipe = build_pipeline(model_name, mode, spw)
                best_pipe.set_params(**best_params)
                best_pipe.fit(X_tr, y_tr)

                p_val_raw = best_pipe.predict_proba(X_val)[:, 1]
                platt = fit_platt_scaler(p_val_raw, y_val.values)

                p_val_cal = apply_platt_scaler(platt, p_val_raw)
                thr, thr_info = choose_threshold_recall_first(
                    y_val.values, p_val_cal, max_pred_pos_rate=MAX_PRED_POS_RATE
                )

                # ----- (2) Refit model on FULL outer train for stronger final model -----
                X_train_full = X.iloc[train_idx]
                y_train_full = y.iloc[train_idx]

                best_pipe.fit(X_train_full, y_train_full)

                # ----- (3) Test with refit model + same platt + same threshold -----
                p_test_raw = best_pipe.predict_proba(X_test)[:, 1]
                p_test_cal = apply_platt_scaler(platt, p_test_raw)

                y_hat = (p_test_cal >= thr).astype(int)

                roc_auc = roc_auc_score(y_test, p_test_cal)
                pr_auc = average_precision_score(y_test, p_test_cal)
                brier = brier_score_loss(y_test, p_test_cal)

                prec = precision_score(y_test, y_hat, zero_division=0)
                rec = recall_score(y_test, y_hat, zero_division=0)
                f1 = f1_score(y_test, y_hat, zero_division=0)
                f2 = fbeta_score(y_test, y_hat, beta=2.0, zero_division=0)

                tn, fp, fn, tp = confusion_matrix(y_test, y_hat).ravel()
                spec = tn / (tn + fp) if (tn + fp) else 0.0
                pred_pos_rate = float(np.mean(y_hat))

                rows.append({
                    "cohort": COHORT, "block": CHOSEN_BLOCK, "repeat": rep+1, "fold": fold,
                    "model": model_name, "mode": mode,
                    "roc_auc": roc_auc, "pr_auc": pr_auc, "brier": brier,
                    "precision": prec, "recall": rec, "specificity": spec,
                    "f1": f1, "f2": f2, "thr": thr,
                    "thr_policy": thr_info.get("policy", "unknown"),
                    "pred_pos_rate": pred_pos_rate
                })

                print(
                    f"Rep {rep+1} Fold {fold:02d} | thr={thr:.2f} ({thr_info.get('policy')}) | "
                    f"PR-AUC={pr_auc:.3f} ROC-AUC={roc_auc:.3f} Brier={brier:.3f} | "
                    f"R={rec:.3f} P={prec:.3f} | Pred+={pred_pos_rate:.3f}"
                )

results_df = pd.DataFrame(rows)

summary = (
    results_df.groupby(["cohort", "block", "model", "mode"])
    .agg(
        roc_auc_mean=("roc_auc", "mean"),
        pr_auc_mean=("pr_auc", "mean"),
        brier_mean=("brier", "mean"),
        recall_mean=("recall", "mean"),
        precision_mean=("precision", "mean"),
        f2_mean=("f2", "mean"),
        thr_mean=("thr", "mean"),
        pred_pos_rate_mean=("pred_pos_rate", "mean"),
        folds=("thr", "count"),
        thr_policy_mode=("thr_policy", lambda s: s.value_counts().index[0] if len(s) else "NA")
    )
    .reset_index()
    .sort_values("pr_auc_mean", ascending=False)
)

print("\n=== SUMMARY ===")
print(summary.to_string(index=False))

Loaded: antepartum shape: (1430, 22)
Sample columns: ['match_key', 'antepartum__depression', 'age', 'epds_antepartum', 'anxious_a', 'avoidant_a', 'reframing', 'selfdistraction', 'venting', 'use_of_informational', 'active', 'denial', 'religion', 'humour', 'disengagement', 'emotional_s', 'substance', 'acceptance', 'planning', 'selfblame', 'women_without_antepartum__depression', 'id']
Detected EPDS: epds_antepartum | cutoff: 12
y distribution: Counter({0: 1196, 1: 234})
Positive prevalence: 0.164 | AP baseline≈prevalence

[GROUP-ID AUTO PICK] Top candidates:
            col  non_null_rate  unique_rate  miss_gap  n_unique  id_boost
     avoidant_a       1.000000     0.048951   0.00000        70       1.0
      match_key       1.000000     0.997203   0.00000      1426       0.0
            age       0.931469     0.025175   0.08194        36       0.0
selfdistraction       0.931469     0.005594   0.08194         8       0.0
    emotional_s       0.931469     0.005594   0.08194         8     

# 7

In [8]:
# 7============================================================
# FINAL PUBLISHABLE PIPELINE
# Perinatal Depression Prediction (Leak-Safe, CV-Stable)
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import re

from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.calibration import CalibratedClassifierCV

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score,
    fbeta_score, brier_score_loss, make_scorer
)

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# ============================================================
# 1) CONFIG
# ============================================================

COHORT = "antepartum"  # "antepartum" or "postpartum"
ANTE_PATH = "/kaggle/input/datasets/maishaidaralkhateeb/ppd-depression/Antepartum_Merged_from_Postpartum.xlsx"
POST_PATH = "/kaggle/input/ppd-depression/Postpartum Depression.xlsx"

EPDS_CUTOFF = 12 if COHORT == "antepartum" else 9

RANDOM_STATE = 42
N_SPLITS = 5
N_REPEATS = 5

# ============================================================
# 2) LOAD + CLEAN
# ============================================================

def clean_columns(df):
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace(r"[^0-9a-zA-Z_]", "", regex=True)
    )
    df = df.loc[:, ~df.columns.duplicated()]
    return df

df = pd.read_excel(ANTE_PATH) if COHORT == "antepartum" else pd.read_csv(POST_PATH)
df = clean_columns(df)
df = df.drop_duplicates()

print("Loaded shape:", df.shape)

# ============================================================
# 3) LABEL CREATION (STRICT)
# ============================================================

epds_col = [c for c in df.columns if "epds" in c and COHORT in c]

if not epds_col:
    epds_col = [c for c in df.columns if "epds" in c]

if not epds_col:
    raise ValueError("No EPDS column found.")

epds_col = epds_col[0]

df[epds_col] = pd.to_numeric(df[epds_col], errors="coerce")
y = (df[epds_col] >= EPDS_CUTOFF).astype(int)

print("Prevalence:", round(y.mean(), 3))

# ============================================================
# 4) HARD LEAKAGE PURGE
# ============================================================

LEAK_PATTERNS = [
    "epds", "depress", "depression",
    "without", "outcome", "label",
    "status", "case", "control",
    "unnamed", "id", "score"
]

drop_cols = [c for c in df.columns if any(p in c for p in LEAK_PATTERNS)]
X = df.drop(columns=drop_cols, errors="ignore").copy()

# numeric only
for c in X.columns:
    X[c] = pd.to_numeric(X[c], errors="coerce")

X = X.dropna(axis=1, how="all")

mask = X.notna().any(axis=1) & y.notna()
X = X.loc[mask]
y = y.loc[mask]

print("Final feature count:", X.shape[1])

# ============================================================
# 5) MODEL PIPELINE (ElasticNet Logistic Regression)
# ============================================================

def build_pipeline(smote=False):

    base_model = LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        l1_ratio=0.5,
        max_iter=5000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    )

    steps = [
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ]

    if smote:
        steps.append(("smote", SMOTE(sampling_strategy=0.5, random_state=RANDOM_STATE)))

    steps.append(("clf", base_model))

    return ImbPipeline(steps) if smote else Pipeline(steps)

param_dist = {
    "clf__C": np.logspace(-3, 2, 15),
    "clf__l1_ratio": [0.25, 0.5, 0.75, 1.0]
}

# ============================================================
# 6) CROSS-VALIDATION WITH INNER THRESHOLDING
# ============================================================

outer_cv = RepeatedStratifiedKFold(
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    random_state=RANDOM_STATE
)

f2_scorer = make_scorer(fbeta_score, beta=2.0)

results = []

for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), 1):

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Inner CV for hyperparameter search
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

    pipe = build_pipeline(smote=True)

    search = RandomizedSearchCV(
        pipe,
        param_dist,
        n_iter=20,
        scoring=f2_scorer,
        cv=inner_cv,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    search.fit(X_train, y_train)
    best_model = search.best_estimator_

    # Calibration (within fold)
    calibrated = CalibratedClassifierCV(best_model, method="sigmoid", cv=3)
    calibrated.fit(X_train, y_train)

    # Validation split for threshold selection
    val_split = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    val_train_idx, val_idx = next(val_split.split(X_train, y_train))

    X_val = X_train.iloc[val_idx]
    y_val = y_train.iloc[val_idx]

    p_val = calibrated.predict_proba(X_val)[:, 1]

    # F2-optimal threshold
    thresholds = np.linspace(0.05, 0.95, 91)
    best_t = 0.5
    best_score = -1

    for t in thresholds:
        score = fbeta_score(y_val, (p_val >= t).astype(int), beta=2.0)
        if score > best_score:
            best_score = score
            best_t = t

    # Final evaluation
    p_test = calibrated.predict_proba(X_test)[:, 1]
    y_pred = (p_test >= best_t).astype(int)

    results.append({
        "roc_auc": roc_auc_score(y_test, p_test),
        "pr_auc": average_precision_score(y_test, p_test),
        "brier": brier_score_loss(y_test, p_test),
        "recall": recall_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "f2": fbeta_score(y_test, y_pred, beta=2.0)
    })

# ============================================================
# 7) SUMMARY
# ============================================================

results_df = pd.DataFrame(results)

summary = results_df.agg(["mean", "std"])

print("\n=== FINAL RESULTS (mean ± std) ===")
print(summary)

Loaded shape: (1430, 22)
Prevalence: 0.164
Final feature count: 16

=== FINAL RESULTS (mean ± std) ===
       roc_auc    pr_auc     brier    recall  precision        f1        f2
mean  0.787415  0.478017  0.112666  0.760148   0.285510  0.410902  0.563884
std   0.042921  0.082726  0.008190  0.138847   0.025485  0.032006  0.065375


# 8

In [9]:
# 8 ============================================================
# FIXED: SHAP Comparison for ALL Models (Leak-safe, Static plots)
# - Robust to SHAP outputs: list / 3D / Explanation
# - FIXES XGBoost TreeExplainer base_score parsing bug
#   -> uses shap.Explainer(model_output="probability") or Permutation fallback
# ============================================================

import os, re, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import shap
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

ANTE_PATH = "/kaggle/input/datasets/maishaidaralkhateeb/ppd-depression/Antepartum_Merged_from_Postpartum.xlsx"
POST_PATH = "/kaggle/input/ppd-depression/Postpartum Depression.xlsx"

OUT_DIR = "/kaggle/working/ppd_outputs"
PLOT_DIR = os.path.join(OUT_DIR, "shap_plots_all_models")
os.makedirs(PLOT_DIR, exist_ok=True)

COHORT = "antepartum"  # "antepartum" or "postpartum"
CUTOFF = 12 if COHORT == "antepartum" else 9

LEAK_PATTERNS = [
    "epds", "depress", "depression",
    "without", "outcome", "label",
    "status", "case", "control",
    "unnamed", "id", "score"
]

def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace(r"[^0-9a-zA-Z_]", "", regex=True)
    )
    df = df.loc[:, ~df.columns.duplicated()].copy()
    return df

def detect_epds_col(df: pd.DataFrame, cohort: str) -> str:
    cols = df.columns.tolist()
    hits = [c for c in cols if ("epds" in c and cohort in c)]
    if not hits:
        hits = [c for c in cols if "epds" in c]
    if not hits:
        raise ValueError("No EPDS column found.")
    return hits[0]

def build_clean_xy(df: pd.DataFrame, cohort: str, cutoff: int):
    df = clean_columns(df)
    epds_col = detect_epds_col(df, cohort)
    df[epds_col] = pd.to_numeric(df[epds_col], errors="coerce")
    y = (df[epds_col] >= cutoff).astype(int)

    drop_cols = [c for c in df.columns if any(p in c for p in LEAK_PATTERNS)]
    X = df.drop(columns=drop_cols, errors="ignore").copy()

    for c in X.columns:
        X[c] = pd.to_numeric(X[c], errors="coerce")
    X = X.dropna(axis=1, how="all")

    mask = X.notna().any(axis=1) & y.notna()
    return X.loc[mask].copy(), y.loc[mask].copy()

df = pd.read_excel(ANTE_PATH) if COHORT == "antepartum" else pd.read_excel(POST_PATH)
X, y = build_clean_xy(df, COHORT, CUTOFF)
print("Cohort:", COHORT, "| X shape:", X.shape, "| prevalence:", round(float(y.mean()), 3))

# Optional heavy libs
HAS_XGB = False
HAS_CAT = False
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

try:
    from catboost import CatBoostClassifier
    HAS_CAT = True
except Exception:
    HAS_CAT = False

USE_SVM_KERNEL = False  # KernelExplainer is slow

MODELS = {}

MODELS["ElasticNet"] = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(
        penalty="elasticnet", solver="saga",
        l1_ratio=0.5, C=0.5,
        class_weight="balanced",
        max_iter=6000,
        random_state=RANDOM_STATE
    ))
])

MODELS["LogReg"] = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(
        solver="liblinear",
        class_weight="balanced",
        max_iter=4000,
        random_state=RANDOM_STATE
    ))
])

MODELS["RF"] = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("clf", RandomForestClassifier(
        n_estimators=400,
        max_depth=5,
        min_samples_leaf=3,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

if HAS_XGB:
    MODELS["XGBoost"] = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("clf", XGBClassifier(
            n_estimators=400,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])

if HAS_CAT:
    MODELS["CatBoost"] = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("clf", CatBoostClassifier(
            iterations=400,
            depth=5,
            learning_rate=0.05,
            verbose=False,
            random_state=RANDOM_STATE,
            class_weights=[1, 2]
        ))
    ])

if USE_SVM_KERNEL:
    MODELS["SVM"] = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("clf", SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=RANDOM_STATE))
    ])

print("Models:", list(MODELS.keys()))

# -------------------------------
# Robust SHAP output normalizer
# -------------------------------
def to_2d_shap_values(shap_out):
    if hasattr(shap_out, "values"):
        shap_out = shap_out.values
    if isinstance(shap_out, list):
        shap_out = shap_out[1] if len(shap_out) > 1 else shap_out[0]
    shap_out = np.array(shap_out)
    if shap_out.ndim == 3:
        shap_out = shap_out[..., 1] if shap_out.shape[-1] >= 2 else shap_out[..., 0]
    if shap_out.ndim != 2:
        raise ValueError(f"Unexpected SHAP shape after normalization: {shap_out.shape}")
    return shap_out

# -------------------------------
# SHAP routing
# -------------------------------
BG_N = min(300, len(X))
EXPLAIN_N = min(800, len(X))

bg_idx = rng.choice(len(X), size=BG_N, replace=False)
ex_idx = rng.choice(len(X), size=EXPLAIN_N, replace=False)

X_bg_raw = X.iloc[bg_idx].copy()
X_ex_raw = X.iloc[ex_idx].copy()

def shap_for_model(name: str, pipe: Pipeline, X_bg: pd.DataFrame, X_ex: pd.DataFrame):
    pipe.fit(X, y)
    steps = pipe.named_steps
    clf = steps.get("clf")

    # Linear (LR / ElasticNet)
    if isinstance(clf, LogisticRegression):
        imputer = steps["impute"]
        scaler = steps["scale"]
        X_bg_s = scaler.transform(imputer.transform(X_bg))
        X_ex_s = scaler.transform(imputer.transform(X_ex))
        masker = shap.maskers.Independent(X_bg_s)
        explainer = shap.LinearExplainer(clf, masker=masker, link=shap.links.logit)
        sv = explainer(X_ex_s)
        return to_2d_shap_values(sv), X_ex

    # Tree: RF / CatBoost
    if clf.__class__.__name__ in ["RandomForestClassifier", "CatBoostClassifier"]:
        imputer = steps["impute"]
        X_bg_i = imputer.transform(X_bg)
        X_ex_i = imputer.transform(X_ex)
        explainer = shap.TreeExplainer(clf)
        sv = explainer.shap_values(X_ex_i)
        X_ex_i_df = pd.DataFrame(X_ex_i, columns=X.columns)
        return to_2d_shap_values(sv), X_ex_i_df

    # Tree: XGBoost (special handling)
    if clf.__class__.__name__ == "XGBClassifier":
        imputer = steps["impute"]
        X_bg_i = imputer.transform(X_bg)
        X_ex_i = imputer.transform(X_ex)
        X_ex_i_df = pd.DataFrame(X_ex_i, columns=X.columns)

        # Try modern unified API first (avoids base_score parsing bug)
        try:
            explainer = shap.Explainer(clf, X_bg_i, model_output="probability")
            sv = explainer(X_ex_i)
            return to_2d_shap_values(sv), X_ex_i_df
        except Exception as e1:
            print(f"[warn] XGBoost shap.Explainer failed: {e1}")

        # Fallback: PermutationExplainer (always works, slower but reliable)
        masker = shap.maskers.Independent(X_bg_i)
        f = lambda data: clf.predict_proba(data)[:, 1]
        explainer = shap.PermutationExplainer(f, masker, random_state=RANDOM_STATE)
        sv = explainer(X_ex_i, max_evals=2 * X_ex_i.shape[1] + 200)
        return to_2d_shap_values(sv), X_ex_i_df

    # SVM: optional slow KernelExplainer
    if USE_SVM_KERNEL and clf.__class__.__name__ == "SVC":
        imputer = steps["impute"]
        scaler = steps["scale"]
        X_bg_s = scaler.transform(imputer.transform(X_bg))
        X_ex_s = scaler.transform(imputer.transform(X_ex))
        f = lambda z: clf.predict_proba(z)[:, 1]
        explainer = shap.KernelExplainer(f, X_bg_s[:min(100, len(X_bg_s))])
        sv = explainer.shap_values(X_ex_s[:min(200, len(X_ex_s))])
        return to_2d_shap_values(sv), X_ex.iloc[:min(200, len(X_ex))]

    raise ValueError(f"No SHAP route for model: {name}")

def save_shap_plots_and_importance(model_name, shap_values_2d, X_disp, cohort):
    mean_abs = np.abs(shap_values_2d).mean(axis=0)
    if len(mean_abs) != X_disp.shape[1]:
        raise ValueError(f"Mismatch: mean_abs len={len(mean_abs)} vs n_features={X_disp.shape[1]} for {model_name}")

    imp_df = pd.DataFrame({"feature": X_disp.columns, "mean_abs_shap": mean_abs}).sort_values("mean_abs_shap", ascending=False)

    csv_path = os.path.join(OUT_DIR, f"shap_importance_{cohort}_{model_name}.csv")
    imp_df.to_csv(csv_path, index=False)

    bees_path = os.path.join(PLOT_DIR, f"shap_beeswarm_{cohort}_{model_name}.png")
    bar_path  = os.path.join(PLOT_DIR, f"shap_bar_{cohort}_{model_name}.png")

    plt.figure(figsize=(9, 5))
    shap.summary_plot(shap_values_2d, X_disp, show=False, plot_type="dot", max_display=20)
    plt.title(f"{cohort.capitalize()} — {model_name} SHAP Beeswarm (Top 20)")
    plt.tight_layout()
    plt.savefig(bees_path, dpi=200)
    plt.close()

    plt.figure(figsize=(9, 5))
    shap.summary_plot(shap_values_2d, X_disp, show=False, plot_type="bar", max_display=20)
    plt.title(f"{cohort.capitalize()} — {model_name} SHAP Global Importance (Top 20)")
    plt.tight_layout()
    plt.savefig(bar_path, dpi=200)
    plt.close()

    return imp_df, csv_path, bees_path, bar_path

# -------------------------------
# Run SHAP
# -------------------------------
os.makedirs(OUT_DIR, exist_ok=True)
all_rows = []

for name, pipe in MODELS.items():
    print(f"\n=== SHAP: {name} ===")
    sv2, X_disp = shap_for_model(name, pipe, X_bg_raw, X_ex_raw)
    imp_df, csv_path, bees_path, bar_path = save_shap_plots_and_importance(name, sv2, X_disp, COHORT)
    print("Saved:", csv_path)
    print("Saved plots:", bees_path, "|", bar_path)
    all_rows.append(imp_df.assign(model=name))

all_imp = pd.concat(all_rows, ignore_index=True)
all_imp_path = os.path.join(OUT_DIR, f"shap_importance_{COHORT}_ALL_MODELS.csv")
all_imp.to_csv(all_imp_path, index=False)
print("\nSaved combined SHAP table:", all_imp_path)

# -------------------------------
# Cross-model comparison plot
# -------------------------------
TOPK = 10
top_feats = (all_imp.sort_values(["model", "mean_abs_shap"], ascending=[True, False])
             .groupby("model").head(TOPK)["feature"].unique().tolist())

pivot = (all_imp[all_imp["feature"].isin(top_feats)]
         .pivot_table(index="feature", columns="model", values="mean_abs_shap", aggfunc="mean")
         .fillna(0.0))

pivot_norm = pivot.div(pivot.sum(axis=0).replace(0, np.nan), axis=1).fillna(0.0)
feat_order = pivot_norm.mean(axis=1).sort_values(ascending=False).index.tolist()
pivot_norm = pivot_norm.loc[feat_order]

plt.figure(figsize=(10, max(4, 0.35 * len(feat_order))))
plt.imshow(pivot_norm.values, aspect="auto")
plt.xticks(range(pivot_norm.shape[1]), pivot_norm.columns, rotation=30, ha="right")
plt.yticks(range(pivot_norm.shape[0]), pivot_norm.index)
plt.colorbar(label="Normalized mean(|SHAP|) within model")
plt.title(f"{COHORT.capitalize()} — SHAP Feature Importance Comparison (Top {TOPK}/model)")
plt.tight_layout()

cmp_path = os.path.join(PLOT_DIR, f"shap_compare_{COHORT}_top_features.png")
plt.savefig(cmp_path, dpi=200)
plt.close()

print("Saved comparison plot:", cmp_path)
print("\nTop features union:", top_feats)

Cohort: antepartum | X shape: (1430, 16) | prevalence: 0.164
Models: ['ElasticNet', 'LogReg', 'RF', 'XGBoost', 'CatBoost']

=== SHAP: ElasticNet ===
Saved: /kaggle/working/ppd_outputs/shap_importance_antepartum_ElasticNet.csv
Saved plots: /kaggle/working/ppd_outputs/shap_plots_all_models/shap_beeswarm_antepartum_ElasticNet.png | /kaggle/working/ppd_outputs/shap_plots_all_models/shap_bar_antepartum_ElasticNet.png

=== SHAP: LogReg ===
Saved: /kaggle/working/ppd_outputs/shap_importance_antepartum_LogReg.csv
Saved plots: /kaggle/working/ppd_outputs/shap_plots_all_models/shap_beeswarm_antepartum_LogReg.png | /kaggle/working/ppd_outputs/shap_plots_all_models/shap_bar_antepartum_LogReg.png

=== SHAP: RF ===
Saved: /kaggle/working/ppd_outputs/shap_importance_antepartum_RF.csv
Saved plots: /kaggle/working/ppd_outputs/shap_plots_all_models/shap_beeswarm_antepartum_RF.png | /kaggle/working/ppd_outputs/shap_plots_all_models/shap_bar_antepartum_RF.png

=== SHAP: XGBoost ===
[warn] XGBoost shap.Exp

PermutationExplainer explainer: 801it [00:54, 12.84it/s]


Saved: /kaggle/working/ppd_outputs/shap_importance_antepartum_XGBoost.csv
Saved plots: /kaggle/working/ppd_outputs/shap_plots_all_models/shap_beeswarm_antepartum_XGBoost.png | /kaggle/working/ppd_outputs/shap_plots_all_models/shap_bar_antepartum_XGBoost.png

=== SHAP: CatBoost ===
Saved: /kaggle/working/ppd_outputs/shap_importance_antepartum_CatBoost.csv
Saved plots: /kaggle/working/ppd_outputs/shap_plots_all_models/shap_beeswarm_antepartum_CatBoost.png | /kaggle/working/ppd_outputs/shap_plots_all_models/shap_bar_antepartum_CatBoost.png

Saved combined SHAP table: /kaggle/working/ppd_outputs/shap_importance_antepartum_ALL_MODELS.csv
Saved comparison plot: /kaggle/working/ppd_outputs/shap_plots_all_models/shap_compare_antepartum_top_features.png

Top features union: ['anxious_a', 'denial', 'selfblame', 'emotional_s', 'active', 'humour', 'planning', 'age', 'selfdistraction', 'venting', 'reframing', 'disengagement']


# 9

In [10]:
# 9 ============================================================
# ✅ FULL FIXED Kaggle Code
# Block-consistent + TRUE Group-safe Tuning + Leak-safe Calibration
# + TWO Ensembles added (SoftVote Uniform + PR-weighted)
# ============================================================

import warnings, hashlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from collections import Counter

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline as SkPipeline

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    precision_score, recall_score, f1_score, fbeta_score, confusion_matrix, make_scorer
)

from sklearn.model_selection import RandomizedSearchCV
from sklearn.base import clone

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# Optional XGBoost
use_xgboost = False
try:
    from xgboost import XGBClassifier
    use_xgboost = True
except Exception:
    use_xgboost = False

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# -------------------------------
# 0) CONFIG
# -------------------------------
COHORT = "antepartum"   # "antepartum" or "postpartum"
ANTE_PATH = "/kaggle/input/datasets/maishaidaralkhateeb/ppd-depression/Antepartum_Merged_from_Postpartum.xlsx"
POST_PATH = "/kaggle/input/ppd-depression/Postpartum Depression.xlsx"

#OUT_DIR = "/kaggle/working/ppd_outputs"

EPDS_CUTOFFS = {"antepartum": 12, "postpartum": 9}
cutoff = EPDS_CUTOFFS[COHORT]

# If you have a real ID column, put its cleaned name here, e.g. "participant_id".
# If it exists, we will use it for groups. Otherwise, we fallback to row-hash.
GROUP_ID_COL = None

# 🔥 strongly recommended to keep leak patterns broad
LEAK_PATTERNS = ["epds", "depress", "depression", "without", "outcome", "label", "score", "unnamed", "id"]

# block detection thresholds
FULL_BLOCK_MIN_RATE = 0.70
SHORT_BLOCK_MAX_RATE = 0.30

# CV settings
N_SPLITS = 5
N_REPEATS = 3
INNER_VAL_FRACTION = 0.20

# Decision policy (paper-friendly)
TARGET_RECALL = 0.85
MIN_PRECISION = 0.20
BETA_FALLBACK = 2.0

# Search
SEARCH_N_ITER = 15
INNER_SEARCH_SPLITS = 3  # group-safe inner CV splits for RandomizedSearchCV

# -------------------------------
# 1) Load + clean headers
# -------------------------------
def load_and_clean_excel(path: str) -> pd.DataFrame:
    df = pd.read_excel(path, header=0)
    df = df.dropna(axis=1, how="all")
    df.columns = (
        df.columns.astype(str)
        .str.strip().str.lower()
        .str.replace(" ", "_")
        .str.replace(r"[^0-9a-zA-Z_]", "", regex=True)
    )
    df = df.loc[:, ~df.columns.duplicated()].copy()
    df = df.drop_duplicates()
    return df

df = load_and_clean_excel(ANTE_PATH if COHORT == "antepartum" else POST_PATH)
print("Loaded:", COHORT, "shape:", df.shape)
print("Sample columns:", df.columns.tolist()[:15])

# -------------------------------
# 2) Detect EPDS column for label
# -------------------------------
def find_best_match(cols, keywords):
    for kw in keywords:
        hits = [c for c in cols if kw in c]
        if hits:
            return hits[0]
    return None

if COHORT == "antepartum":
    epds_col = find_best_match(df.columns, ["epds_antepartum", "epdsantepartum", "epds"])
else:
    epds_col = find_best_match(df.columns, ["epds_postpartum", "epdspostpartum", "epds"])

if epds_col is None:
    raise ValueError("Could not find EPDS column.")

df[epds_col] = pd.to_numeric(df[epds_col], errors="coerce")
y = (df[epds_col] >= cutoff).astype(int)

print("Detected EPDS:", epds_col, "| cutoff:", cutoff)
print("y distribution:", Counter(y))
print("PR-AUC baseline (prevalence):", round(float(y.mean()), 3))

# -------------------------------
# 3) Build numeric-only candidate feature table
# -------------------------------
drop_cols = [c for c in df.columns if any(p in c for p in LEAK_PATTERNS)]
X_all = df.drop(columns=drop_cols, errors="ignore").copy()

for c in X_all.columns:
    X_all[c] = pd.to_numeric(X_all[c], errors="coerce")
X_all = X_all.dropna(axis=1, how="all")

mask = X_all.notna().any(axis=1) & y.notna()
X_all = X_all.loc[mask].copy()
y = y.loc[mask].copy()

print("\nNumeric feature pool shape:", X_all.shape)
print("y distribution after basic align:", Counter(y))

# -------------------------------
# 4) Identify blocks by coverage (FULL vs SHORT)
# -------------------------------
non_missing_rate = X_all.notna().mean().sort_values(ascending=False)
full_cols = non_missing_rate[non_missing_rate >= FULL_BLOCK_MIN_RATE].index.tolist()
short_cols = non_missing_rate[non_missing_rate <= SHORT_BLOCK_MAX_RATE].index.tolist()

print("\n[BLOCK DETECT] Full block cols:", len(full_cols), "| Short block cols:", len(short_cols))

X_full = X_all[full_cols].copy() if len(full_cols) else pd.DataFrame(index=X_all.index)
X_short = X_all[short_cols].copy() if len(short_cols) else pd.DataFrame(index=X_all.index)

def block_valid(X, y):
    if X.shape[1] == 0:
        return X.copy(), y.copy(), False
    keep = X.notna().sum(axis=1) >= min(3, X.shape[1])
    X2 = X.loc[keep].copy()
    y2 = y.loc[keep].copy()
    return X2, y2, (y2.nunique() == 2)

def missing_gap_score(X, y):
    if X.shape[1] == 0:
        return np.inf
    miss_by_class = X.isna().groupby(y).mean()
    if miss_by_class.shape[0] < 2:
        return np.inf
    gap = (miss_by_class.loc[1] - miss_by_class.loc[0]).abs()
    return float(gap.mean())

Xf, yf, okf = block_valid(X_full, y)
Xs, ys, oks = block_valid(X_short, y)

gap_f = missing_gap_score(Xf, yf) if okf else np.inf
gap_s = missing_gap_score(Xs, ys) if oks else np.inf

print("\n[BLOCK SCORES]")
print("FULL  -> valid:", okf, "| rows:", len(Xf), "| gap_mean:", gap_f)
print("SHORT -> valid:", oks, "| rows:", len(Xs), "| gap_mean:", gap_s)

if gap_f <= gap_s:
    X, y = Xf, yf
    CHOSEN_BLOCK = "FULL"
else:
    X, y = Xs, ys
    CHOSEN_BLOCK = "SHORT"

print(f"\n✅ Chosen block: {CHOSEN_BLOCK}")
print("Final X shape:", X.shape)
print("Final y distribution:", Counter(y))

if y.nunique() < 2:
    raise ValueError("After block selection, only one class remains.")

# -------------------------------
# 5) Groups (prefer real ID if available)
# -------------------------------
def make_groups(df_raw: pd.DataFrame, X: pd.DataFrame) -> pd.Series:
    if GROUP_ID_COL is not None and GROUP_ID_COL in df_raw.columns:
        g = df_raw.loc[X.index, GROUP_ID_COL].astype(str).fillna("NA")
        return pd.Series(g.values, index=X.index)

    # Fallback: hash feature rows (duplicate-guard, not true subject grouping)
    X_for_hash = X.fillna("__MISSING__").astype(str)
    g = X_for_hash.apply(
        lambda r: hashlib.md5("||".join(r.values.tolist()).encode("utf-8")).hexdigest(),
        axis=1
    )
    return pd.Series(g.values, index=X.index)

groups = make_groups(df, X)
print("\n[GROUP CHECK] Unique groups:", groups.nunique(), "out of", len(groups))
print("[GROUP CHECK] Largest group size:", int(groups.value_counts().max()))

# -------------------------------
# 6) Splitters (outer + inner group-safe)
# -------------------------------
try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAS_SGKF = True
except Exception:
    HAS_SGKF = False
    from sklearn.model_selection import GroupKFold

def outer_splitter(seed: int):
    if HAS_SGKF:
        cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
        return cv.split(X, y, groups=groups)
    else:
        gkf = GroupKFold(n_splits=N_SPLITS)
        return gkf.split(X, y, groups=groups)

def group_safe_inner_split(train_idx, val_frac, seed):
    rng = np.random.default_rng(seed)
    train_groups = groups.iloc[train_idx]
    uniq_g = train_groups.unique()
    rng.shuffle(uniq_g)
    n_val_groups = max(1, int(len(uniq_g) * val_frac))
    val_g = set(uniq_g[:n_val_groups])
    inner_val_mask = train_groups.isin(val_g).values
    inner_val_idx = train_idx[inner_val_mask]
    inner_train_idx = train_idx[~inner_val_mask]
    return inner_train_idx, inner_val_idx

def make_group_cv(seed: int, n_splits: int):
    if HAS_SGKF:
        return StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    else:
        return GroupKFold(n_splits=n_splits)

# -------------------------------
# 7) Models + pipelines
# -------------------------------
def make_base_estimator(model_name, mode, spw):
    if model_name == "LR":
        cw = "balanced" if mode == "weighted" else None
        return LogisticRegression(max_iter=5000, class_weight=cw, random_state=RANDOM_STATE)

    if model_name == "SVM":
        cw = "balanced" if mode == "weighted" else None
        return SVC(kernel="rbf", probability=True, class_weight=cw, random_state=RANDOM_STATE)

    if model_name == "RF":
        cw = "balanced" if mode == "weighted" else None
        return RandomForestClassifier(n_estimators=400, class_weight=cw, random_state=RANDOM_STATE, n_jobs=-1)

    if model_name == "XGB":
        if not use_xgboost:
            return HistGradientBoostingClassifier(random_state=RANDOM_STATE)

        spw2 = spw if mode == "weighted" else 1.0
        return XGBClassifier(
            n_estimators=500, max_depth=3, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
            scale_pos_weight=spw2, eval_metric="logloss",
            random_state=RANDOM_STATE, n_jobs=-1
        )

    return HistGradientBoostingClassifier(random_state=RANDOM_STATE)

def build_pipeline(model_name, mode, spw):
    imputer = ("imputer", SimpleImputer(strategy="median"))
    scaler = ("scaler", StandardScaler())
    estimator = ("model", make_base_estimator(model_name, mode, spw))

    if mode == "resampled":
        smote = ("smote", SMOTE(sampling_strategy=0.8, random_state=RANDOM_STATE))
        under = ("under", RandomUnderSampler(sampling_strategy=1.0, random_state=RANDOM_STATE))
        return ImbPipeline([imputer, scaler, smote, under, estimator])

    return SkPipeline([imputer, scaler, estimator])

def get_param_distributions(model_name):
    if model_name == "LR":
        return {"model__C": np.logspace(-3, 3, 25)}
    if model_name == "SVM":
        return {"model__C": np.logspace(-2, 3, 20), "model__gamma": ["scale", "auto"]}
    if model_name == "RF":
        return {"model__max_depth": [None, 4, 7, 12], "model__min_samples_leaf": [1, 2, 4]}
    if model_name == "XGB":
        return {"model__max_depth": [2, 3, 4], "model__learning_rate": [0.03, 0.05, 0.1]}
    return {}

# -------------------------------
# 8) Threshold selection (recall-first + precision floor)
# -------------------------------
def choose_threshold_recall_first(y_val, p_val,
                                 target_recall=TARGET_RECALL,
                                 min_precision=MIN_PRECISION,
                                 beta_fallback=BETA_FALLBACK):
    thresholds = np.linspace(0.01, 0.99, 99)
    best = None  # (recall, precision, f1, thr)

    for t in thresholds:
        y_hat = (p_val >= t).astype(int)
        rec = recall_score(y_val, y_hat, zero_division=0)
        prec = precision_score(y_val, y_hat, zero_division=0)
        if rec >= target_recall and prec >= min_precision:
            f1 = f1_score(y_val, y_hat, zero_division=0)
            cand = (rec, prec, f1, t)
            if best is None or cand > best:
                best = cand

    if best is not None:
        rec, prec, f1, t = best
        return float(t), {"policy": "recall+precision", "val_recall": rec, "val_precision": prec, "val_f1": f1}

    # fallback: maximize F-beta
    best_t, best_score = 0.5, -1.0
    for t in thresholds:
        y_hat = (p_val >= t).astype(int)
        score = fbeta_score(y_val, y_hat, beta=beta_fallback, zero_division=0)
        if score > best_score:
            best_score = score
            best_t = t

    return float(best_t), {"policy": f"fallback_f{beta_fallback}", "val_fbeta": best_score}

# -------------------------------
# 9) Leak-safe calibration using inner-val only (Platt scaling)
# -------------------------------
def fit_platt_scaler(p_val: np.ndarray, y_val: np.ndarray):
    p_val = np.clip(p_val, 1e-6, 1 - 1e-6).reshape(-1, 1)
    cal = LogisticRegression(solver="lbfgs", max_iter=2000)
    cal.fit(p_val, y_val.astype(int))
    return cal

def apply_platt_scaler(cal_model, p: np.ndarray) -> np.ndarray:
    p = np.clip(p, 1e-6, 1 - 1e-6).reshape(-1, 1)
    return cal_model.predict_proba(p)[:, 1]

# -------------------------------
# 9B) Ensemble helper functions
# -------------------------------
def get_probs(best_pipe, X_any):
    if hasattr(best_pipe, "predict_proba"):
        return best_pipe.predict_proba(X_any)[:, 1]
    s = best_pipe.decision_function(X_any)
    return (s - s.min()) / (s.max() - s.min() + 1e-12)

def soft_vote(probs_list, weights=None):
    P = np.vstack(probs_list)
    if weights is None:
        w = np.ones(P.shape[0]) / P.shape[0]
    else:
        w = np.asarray(weights, dtype=float)
        w = w / (w.sum() + 1e-12)
    return (w[:, None] * P).sum(axis=0)

def compute_pr_auc_weight(pipe_fitted, X_tr, y_tr, g_tr, seed):
    """Group-safe CV PR-AUC on inner-train only (for ensemble weights)."""
    inner_cv = make_group_cv(seed=seed, n_splits=INNER_SEARCH_SPLITS)
    prs = []
    for tr2, te2 in inner_cv.split(X_tr, y_tr, groups=g_tr):
        m = clone(pipe_fitted)
        m.fit(X_tr.iloc[tr2], y_tr.iloc[tr2])
        p = get_probs(m, X_tr.iloc[te2])
        prs.append(average_precision_score(y_tr.iloc[te2], p))
    return float(np.mean(prs)) if prs else 1e-6

def pack_metrics(tag, mode_tag, y_true, p_prob, thr_used, thr_policy):
    y_hat = (p_prob >= thr_used).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_hat).ravel()
    spec = tn / (tn + fp) if (tn + fp) else 0.0
    return {
        "cohort": COHORT,
        "block": CHOSEN_BLOCK,
        "model": tag,
        "mode": mode_tag,
        "roc_auc": roc_auc_score(y_true, p_prob),
        "pr_auc": average_precision_score(y_true, p_prob),
        "brier": brier_score_loss(y_true, p_prob),
        "precision": precision_score(y_true, y_hat, zero_division=0),
        "recall": recall_score(y_true, y_hat, zero_division=0),
        "specificity": spec,
        "f1": f1_score(y_true, y_hat, zero_division=0),
        "f2": fbeta_score(y_true, y_hat, beta=2.0, zero_division=0),
        "thr": float(thr_used),
        "thr_policy": thr_policy
    }

# -------------------------------
# 10) Shuffle sanity test (quick)
# -------------------------------
def quick_groupcv_auc(seed=42, shuffle_labels=False):
    rng = np.random.default_rng(seed)
    y_tested = y.copy()
    if shuffle_labels:
        y_tested = pd.Series(rng.permutation(y_tested.values), index=y_tested.index)

    aucs, aps = [], []
    for tr_idx, te_idx in outer_splitter(seed):
        X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
        y_tr, y_te = y_tested.iloc[tr_idx], y_tested.iloc[te_idx]
        n_pos = int((y_tr == 1).sum()); n_neg = int((y_tr == 0).sum())
        spw = (n_neg / max(n_pos, 1))

        pipe = build_pipeline("LR", "weighted", spw)
        pipe.fit(X_tr, y_tr)
        p = pipe.predict_proba(X_te)[:, 1]

        aucs.append(roc_auc_score(y_te, p))
        aps.append(average_precision_score(y_te, p))

    return float(np.mean(aucs)), float(np.mean(aps))

print("\n[SHUFFLE SANITY]")
ra, rp = quick_groupcv_auc(seed=42, shuffle_labels=False)
sa, sp = quick_groupcv_auc(seed=42, shuffle_labels=True)
print(f"Real     -> ROC-AUC={ra:.3f}, PR-AUC={rp:.3f}")
print(f"Shuffled -> ROC-AUC={sa:.3f}, PR-AUC={sp:.3f}")

# -------------------------------
# 11) Full evaluation (+ TWO ensembles)
# -------------------------------
model_specs = ["LR", "SVM", "RF", "XGB"]
modes = ["weighted", "resampled"]
f2_scorer = make_scorer(fbeta_score, beta=2.0, zero_division=0)

rows = []

for rep in range(N_REPEATS):
    rep_seed = RANDOM_STATE + rep
    print(f"\n===== REPEAT {rep+1}/{N_REPEATS} | seed={rep_seed} =====")

    for model_name in model_specs:
        for mode in modes:
            print(f"\n=== {COHORT.upper()} | MODEL={model_name} | MODE={mode} ===")

            for fold, (train_idx, test_idx) in enumerate(outer_splitter(rep_seed), start=1):

                X_test = X.iloc[test_idx]
                y_test = y.iloc[test_idx]

                # inner train/val split (group-safe)
                inner_train_idx, inner_val_idx = group_safe_inner_split(
                    train_idx, INNER_VAL_FRACTION, seed=rep_seed + 1000 + fold
                )

                X_tr = X.iloc[inner_train_idx]
                y_tr = y.iloc[inner_train_idx]
                g_tr = groups.iloc[inner_train_idx]

                X_val = X.iloc[inner_val_idx]
                y_val = y.iloc[inner_val_idx]

                n_pos = int((y_tr == 1).sum()); n_neg = int((y_tr == 0).sum())
                spw = (n_neg / max(n_pos, 1))

                pipe = build_pipeline(model_name, mode, spw)
                param_dist = get_param_distributions(model_name)
                inner_cv = make_group_cv(seed=rep_seed + 2000 + fold, n_splits=INNER_SEARCH_SPLITS)

                search = RandomizedSearchCV(
                    estimator=pipe,
                    param_distributions=param_dist,
                    n_iter=SEARCH_N_ITER,
                    scoring=f2_scorer,
                    cv=inner_cv,
                    random_state=rep_seed,
                    n_jobs=-1,
                    refit=True
                )
                search.fit(X_tr, y_tr, groups=g_tr)

                best_pipe = search.best_estimator_
                best_pipe.fit(X_tr, y_tr)

                # Platt calibration on inner-val only
                p_val_raw = get_probs(best_pipe, X_val)
                platt = fit_platt_scaler(p_val_raw, y_val.values)

                p_val_cal = apply_platt_scaler(platt, p_val_raw)
                thr, thr_info = choose_threshold_recall_first(y_val.values, p_val_cal)

                # Evaluate base model on test (calibrated)
                p_test_raw = get_probs(best_pipe, X_test)
                p_test_cal = apply_platt_scaler(platt, p_test_raw)

                rows.append({
                    **pack_metrics(model_name, mode, y_test, p_test_cal, thr, thr_info.get("policy","")),
                    "repeat": rep+1, "fold": fold
                })

                print(
                    f"Rep {rep+1} Fold {fold:02d} | BASE={model_name}/{mode} thr={thr:.2f} | "
                    f"PR-AUC={average_precision_score(y_test, p_test_cal):.3f} ROC-AUC={roc_auc_score(y_test, p_test_cal):.3f}"
                )

                # ============================================================
                # ✅ TWO ensembles computed ONCE per fold:
                # Use tuned LR(weighted) + tuned RF(weighted) with their own Platt
                # Trigger only on the first base loop to avoid duplicates
                # ============================================================
                if model_name == "LR" and mode == "weighted":
                    # ---- Base A = already tuned LR(weighted) in this block ----
                    pipe_A = best_pipe
                    platt_A = platt
                    p_val_A = apply_platt_scaler(platt_A, get_probs(pipe_A, X_val))
                    p_test_A = apply_platt_scaler(platt_A, get_probs(pipe_A, X_test))

                    # ---- Base B = tuned RF(weighted) (group-safe) ----
                    baseB_name = "RF"
                    baseB_mode = "weighted"
                    pipeB = build_pipeline(baseB_name, baseB_mode, spw)
                    paramB = get_param_distributions(baseB_name)
                    inner_cvB = make_group_cv(seed=rep_seed + 3000 + fold, n_splits=INNER_SEARCH_SPLITS)

                    searchB = RandomizedSearchCV(
                        estimator=pipeB,
                        param_distributions=paramB,
                        n_iter=max(8, SEARCH_N_ITER // 2),
                        scoring=f2_scorer,
                        cv=inner_cvB,
                        random_state=rep_seed + 77,
                        n_jobs=-1,
                        refit=True
                    )
                    searchB.fit(X_tr, y_tr, groups=g_tr)
                    pipe_B = searchB.best_estimator_
                    pipe_B.fit(X_tr, y_tr)

                    p_val_B_raw = get_probs(pipe_B, X_val)
                    platt_B = fit_platt_scaler(p_val_B_raw, y_val.values)

                    p_val_B = apply_platt_scaler(platt_B, p_val_B_raw)
                    p_test_B = apply_platt_scaler(platt_B, get_probs(pipe_B, X_test))

                    # -----------------------------
                    # Ensemble 1: SoftVote Uniform
                    # -----------------------------
                    p_val_u = soft_vote([p_val_A, p_val_B], weights=None)
                    thr_u, info_u = choose_threshold_recall_first(y_val.values, p_val_u)
                    p_test_u = soft_vote([p_test_A, p_test_B], weights=None)

                    rows.append({
                        **pack_metrics("Ensemble_SoftVote_Uniform", "ensemble", y_test, p_test_u, thr_u, info_u.get("policy","")),
                        "repeat": rep+1, "fold": fold
                    })

                    # -----------------------------
                    # Ensemble 2: SoftVote Weighted
                    # weights from group-safe CV PR-AUC on inner-train
                    # -----------------------------
                    wA = compute_pr_auc_weight(pipe_A, X_tr, y_tr, g_tr, seed=rep_seed + 4000 + fold)
                    wB = compute_pr_auc_weight(pipe_B, X_tr, y_tr, g_tr, seed=rep_seed + 5000 + fold)

                    p_val_w = soft_vote([p_val_A, p_val_B], weights=[wA, wB])
                    thr_w, info_w = choose_threshold_recall_first(y_val.values, p_val_w)
                    p_test_w = soft_vote([p_test_A, p_test_B], weights=[wA, wB])

                    rows.append({
                        **pack_metrics("Ensemble_SoftVote_Weighted", "ensemble", y_test, p_test_w, thr_w, info_w.get("policy","")),
                        "repeat": rep+1, "fold": fold
                    })

                    print(
                        f"   + Ensembles | Uniform(PR-AUC={average_precision_score(y_test, p_test_u):.3f}) "
                        f"| Weighted(PR-AUC={average_precision_score(y_test, p_test_w):.3f})"
                    )

# -------------------------------
# 12) Summary
# -------------------------------
results_df = pd.DataFrame(rows)

summary = (
    results_df.groupby(["cohort", "block", "model", "mode"])
    .agg(
        roc_auc_mean=("roc_auc", "mean"),
        pr_auc_mean=("pr_auc", "mean"),
        brier_mean=("brier", "mean"),
        recall_mean=("recall", "mean"),
        precision_mean=("precision", "mean"),
        f2_mean=("f2", "mean"),
        thr_mean=("thr", "mean"),
        folds=("thr", "count"),
        thr_policy_mode=("thr_policy", lambda s: s.value_counts().index[0] if len(s) else "NA")
    )
    .reset_index()
    .sort_values("pr_auc_mean", ascending=False)
)

print("\n=== SUMMARY (Base models + TWO ensembles) ===")
print(summary.to_string(index=False))

# Save outputs (optional)
results_df.to_csv(f"ppd_{COHORT}_folds_with_ensembles.csv", index=False)
summary.to_csv(f"ppd_{COHORT}_summary_with_ensembles.csv", index=False)
print("\nSaved:")
print(f"- ppd_{COHORT}_folds_with_ensembles.csv")
print(f"- ppd_{COHORT}_summary_with_ensembles.csv")

Loaded: antepartum shape: (1430, 22)
Sample columns: ['match_key', 'antepartum__depression', 'age', 'epds_antepartum', 'anxious_a', 'avoidant_a', 'reframing', 'selfdistraction', 'venting', 'use_of_informational', 'active', 'denial', 'religion', 'humour', 'disengagement']
Detected EPDS: epds_antepartum | cutoff: 12
y distribution: Counter({0: 1196, 1: 234})
PR-AUC baseline (prevalence): 0.164

Numeric feature pool shape: (1430, 16)
y distribution after basic align: Counter({0: 1196, 1: 234})

[BLOCK DETECT] Full block cols: 16 | Short block cols: 0

[BLOCK SCORES]
FULL  -> valid: True | rows: 1430 | gap_mean: 0.020537207357859532
SHORT -> valid: False | rows: 1430 | gap_mean: inf

✅ Chosen block: FULL
Final X shape: (1430, 16)
Final y distribution: Counter({0: 1196, 1: 234})

[GROUP CHECK] Unique groups: 1429 out of 1430
[GROUP CHECK] Largest group size: 2

[SHUFFLE SANITY]
Real     -> ROC-AUC=0.797, PR-AUC=0.473
Shuffled -> ROC-AUC=0.464, PR-AUC=0.152

===== REPEAT 1/3 | seed=42 =====


# 10

In [11]:
# ============================================================
# #10 — Proper statistical comparison: Antepartum vs Postpartum (FIXED IO)
# - Handles XLSX vs CSV automatically + CSV encoding fallbacks
# - Prevalence chi-square + risk difference
# - Shared feature tests (Mann–Whitney + Cohen's d)
# - Model performance comparison (CV distributions + bootstrap CI)
# - Saves CSV outputs to /mnt/data/ppd_outputs/
# ============================================================

import warnings, re, os
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from scipy.stats import chi2_contingency, mannwhitneyu
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

RANDOM_STATE = 42

# ✅ Use your actual uploaded files in this chat environment
ANTE_PATH = "/kaggle/input/datasets/maishaidaralkhateeb/ppd-depression/Antepartum_Merged_from_Postpartum.xlsx"
POST_PATH = "/kaggle/input/datasets/maishaidaralkhateeb/ppd-depression/Postpartum Depression.xlsx"

OUT_DIR = "/kaggle/working/ppd_outputs"

os.makedirs(OUT_DIR, exist_ok=True)

LEAK_PATTERNS = [
    "epds", "depress", "depression",
    "without", "outcome", "label",
    "status", "case", "control",
    "unnamed", "id", "score"
]

# -------------------------------
# 0) Smart loader (xlsx/csv + encoding fallbacks)
# -------------------------------
def smart_read_table(path: str) -> pd.DataFrame:
    ext = os.path.splitext(path.lower())[1]

    if ext in [".xlsx", ".xls"]:
        return pd.read_excel(path)

    if ext in [".csv", ".txt"]:
        encodings = ["utf-8", "utf-8-sig", "cp1252", "latin1"]
        last_err = None
        for enc in encodings:
            try:
                return pd.read_csv(path, encoding=enc)
            except UnicodeDecodeError as e:
                last_err = e
                continue
        # fallback: python engine
        for enc in encodings:
            try:
                return pd.read_csv(path, encoding=enc, engine="python")
            except Exception as e:
                last_err = e
                continue
        raise last_err

    raise ValueError(f"Unsupported file type for: {path}")

# -------------------------------
# 1) Cleaning + detection helpers
# -------------------------------
def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace(r"[^0-9a-zA-Z_]", "", regex=True)
    )
    df = df.loc[:, ~df.columns.duplicated()].copy()
    return df

def detect_epds_col(df, cohort):
    hits = [c for c in df.columns if ("epds" in c and cohort in c)]
    if not hits:
        hits = [c for c in df.columns if "epds" in c]
    if not hits:
        raise ValueError("No EPDS column found.")
    return hits[0]

def find_col(df, keys):
    for k in keys:
        for c in df.columns:
            if k in c:
                return c
    return None

# -------------------------------
# 2) Build shared features (same definition in both cohorts)
# -------------------------------
def build_shared_features(df):
    """
    Small, interpretable shared feature set for BOTH cohorts:
      - age
      - anxious_attachment / anxious_a
      - avoidant_attachment / avoidant_a
      - coping composites from available columns
    """
    df = df.copy()

    age_col = find_col(df, ["age"])
    anx_col = find_col(df, ["anxious_attachment", "anxious_a"])
    avo_col = find_col(df, ["avoidant_attachment", "avoidant_a"])

    def safe_num(col):
        if col is None or col not in df.columns:
            return None
        return pd.to_numeric(df[col], errors="coerce")

    X_parts = []
    if age_col: X_parts.append(safe_num(age_col).rename("age"))
    if anx_col: X_parts.append(safe_num(anx_col).rename("anxious_attachment"))
    if avo_col: X_parts.append(safe_num(avo_col).rename("avoidant_attachment"))

    # coping composites (conservative; "contains" matching)
    def mean_of_contains(tokens, newname):
        cols = [c for c in df.columns if any(t in c for t in tokens)]
        if not cols:
            return None
        tmp = df[cols].apply(pd.to_numeric, errors="coerce")
        return tmp.mean(axis=1).rename(newname)

    adaptive = mean_of_contains(["active", "planning", "reframing", "acceptance", "humour", "humor"], "adaptive_coping")
    emotional = mean_of_contains(["emotional", "support", "venting", "selfblame", "self_blame"], "emotional_coping")
    maladaptive = mean_of_contains(["denial", "disengagement", "substance"], "maladaptive_coping")

    for s in [adaptive, emotional, maladaptive]:
        if s is not None:
            X_parts.append(s)

    if not X_parts:
        raise ValueError("Could not build shared feature set.")

    X = pd.concat(X_parts, axis=1)
    # remove dup columns if any
    X = X.loc[:, ~X.columns.duplicated()].copy()
    return X

def build_clean_y_and_sharedX(df, cohort, cutoff):
    df = clean_columns(df)
    epds_col = detect_epds_col(df, cohort)
    df[epds_col] = pd.to_numeric(df[epds_col], errors="coerce")
    y = (df[epds_col] >= cutoff).astype(int)

    # leakage purge (for fairness, remove before building coping)
    drop_cols = [c for c in df.columns if any(p in c for p in LEAK_PATTERNS)]
    df2 = df.drop(columns=drop_cols, errors="ignore")

    X = build_shared_features(df2)

    mask = y.notna() & X.notna().any(axis=1)
    return X.loc[mask].copy(), y.loc[mask].copy()

# -------------------------------
# 3) Stats helpers
# -------------------------------
def cohens_d(a, b):
    a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
    a = a[~np.isnan(a)]; b = b[~np.isnan(b)]
    if len(a) < 2 or len(b) < 2:
        return np.nan
    sa = np.var(a, ddof=1); sb = np.var(b, ddof=1)
    sp = np.sqrt(((len(a)-1)*sa + (len(b)-1)*sb) / max(len(a)+len(b)-2, 1))
    if sp == 0:
        return np.nan
    return (np.mean(a) - np.mean(b)) / sp

def prevalence_stats(yA, yP):
    tab = np.array([
        [(yA == 1).sum(), (yA == 0).sum()],
        [(yP == 1).sum(), (yP == 0).sum()]
    ])
    chi2, p, dof, exp = chi2_contingency(tab)
    prevA = float(yA.mean())
    prevP = float(yP.mean())
    rd = float(prevP - prevA)  # post - ante
    return {
        "prev_ante": prevA,
        "prev_post": prevP,
        "risk_diff_post_minus_ante": rd,
        "chi2": float(chi2),
        "p_value": float(p),
        "table": tab
    }

def compare_features(XA, XP):
    rows = []
    common = [c for c in XA.columns if c in XP.columns]
    for c in common:
        a = XA[c].to_numpy(dtype=float)
        b = XP[c].to_numpy(dtype=float)
        a = a[~np.isnan(a)]
        b = b[~np.isnan(b)]
        if len(a) < 10 or len(b) < 10:
            continue
        _, p = mannwhitneyu(a, b, alternative="two-sided")
        rows.append({
            "feature": c,
            "ante_mean": float(np.mean(a)),
            "post_mean": float(np.mean(b)),
            "ante_median": float(np.median(a)),
            "post_median": float(np.median(b)),
            "cohens_d_post_minus_ante": float(cohens_d(b, a)),
            "mannwhitney_p": float(p)
        })
    out = pd.DataFrame(rows).sort_values("mannwhitney_p", ascending=True)
    return out

def cv_metrics_distribution(X, y, n_splits=5, seed=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    rocs, prs = [], []

    pipe = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(
            penalty="elasticnet", solver="saga",
            l1_ratio=0.5, C=0.5,
            class_weight="balanced",
            max_iter=6000,
            random_state=seed
        ))
    ])

    for tr, te in skf.split(X, y):
        pipe.fit(X.iloc[tr], y.iloc[tr])
        p = pipe.predict_proba(X.iloc[te])[:, 1]
        rocs.append(roc_auc_score(y.iloc[te], p))
        prs.append(average_precision_score(y.iloc[te], p))

    return np.array(rocs), np.array(prs)

def bootstrap_diff(a, b, n_boot=5000, seed=42):
    rng = np.random.default_rng(seed)
    diffs = []
    for _ in range(n_boot):
        sa = rng.choice(a, size=len(a), replace=True)
        sb = rng.choice(b, size=len(b), replace=True)
        diffs.append(np.mean(sb) - np.mean(sa))  # post - ante
    diffs = np.array(diffs)
    lo, hi = np.quantile(diffs, [0.025, 0.975])
    return float(np.mean(diffs)), float(lo), float(hi)

# -------------------------------
# Load cohorts (FIXED)
# -------------------------------
ante_raw = smart_read_table(ANTE_PATH)
post_raw = smart_read_table(POST_PATH)

XA, yA = build_clean_y_and_sharedX(ante_raw, "antepartum", 12)
XP, yP = build_clean_y_and_sharedX(post_raw, "postpartum", 9)

print("Ante X:", XA.shape, "Prevalence:", round(float(yA.mean()), 3))
print("Post X:", XP.shape, "Prevalence:", round(float(yP.mean()), 3))

# -------------------------------
# 1) Prevalence comparison
# -------------------------------
prev_res = prevalence_stats(yA, yP)
print("\n=== Prevalence Comparison ===")
print("Ante:", round(prev_res["prev_ante"], 3), "Post:", round(prev_res["prev_post"], 3))
print("Risk diff (Post - Ante):", round(prev_res["risk_diff_post_minus_ante"], 3))
print("Chi-square p-value:", prev_res["p_value"])
print("2x2 table [[ante pos, ante neg],[post pos, post neg]]:\n", prev_res["table"])

# Save prevalence summary
prev_out = os.path.join(OUT_DIR, "cohort_prevalence_comparison.csv")
pd.DataFrame([{
    "prev_ante": prev_res["prev_ante"],
    "prev_post": prev_res["prev_post"],
    "risk_diff_post_minus_ante": prev_res["risk_diff_post_minus_ante"],
    "chi2": prev_res["chi2"],
    "p_value": prev_res["p_value"],
    "ante_pos": int(prev_res["table"][0,0]),
    "ante_neg": int(prev_res["table"][0,1]),
    "post_pos": int(prev_res["table"][1,0]),
    "post_neg": int(prev_res["table"][1,1]),
}]).to_csv(prev_out, index=False)
print("Saved:", prev_out)

# -------------------------------
# 2) Feature comparisons
# -------------------------------
feat_cmp = compare_features(XA, XP)
out_feat = os.path.join(OUT_DIR, "cohort_feature_comparison.csv")
feat_cmp.to_csv(out_feat, index=False)
print("\n=== Shared Feature Comparison (saved) ===")
print("Saved:", out_feat)
print(feat_cmp.head(10).to_string(index=False))

# -------------------------------
# 3) Model performance comparison (CV distributions + bootstrap CI)
# -------------------------------
rocA, prA = cv_metrics_distribution(XA, yA, n_splits=5, seed=42)
rocP, prP = cv_metrics_distribution(XP, yP, n_splits=5, seed=42)

d_roc, lo_roc, hi_roc = bootstrap_diff(rocA, rocP)
d_pr, lo_pr, hi_pr = bootstrap_diff(prA, prP)

print("\n=== Model Performance Comparison (Post - Ante) ===")
print("ROC-AUC mean Ante:", round(float(rocA.mean()), 3), "Post:", round(float(rocP.mean()), 3))
print("ROC-AUC diff mean:", round(d_roc, 3), "95% CI:", (round(lo_roc, 3), round(hi_roc, 3)))

print("PR-AUC mean Ante:", round(float(prA.mean()), 3), "Post:", round(float(prP.mean()), 3))
print("PR-AUC diff mean:", round(d_pr, 3), "95% CI:", (round(lo_pr, 3), round(hi_pr, 3)))

perf_out = os.path.join(OUT_DIR, "cohort_model_performance_comparison.csv")
pd.DataFrame({
    "metric": ["roc_auc", "pr_auc"],
    "ante_mean": [rocA.mean(), prA.mean()],
    "post_mean": [rocP.mean(), prP.mean()],
    "diff_post_minus_ante_mean": [d_roc, d_pr],
    "diff_95ci_low": [lo_roc, lo_pr],
    "diff_95ci_high": [hi_roc, hi_pr]
}).to_csv(perf_out, index=False)

print("Saved:", perf_out)

Ante X: (1430, 5) Prevalence: 0.164
Post X: (1351, 5) Prevalence: 0.212

=== Prevalence Comparison ===
Ante: 0.164 Post: 0.212
Risk diff (Post - Ante): 0.049
Chi-square p-value: 0.0011633043896988441
2x2 table [[ante pos, ante neg],[post pos, post neg]]:
 [[ 234 1196]
 [ 287 1064]]
Saved: /kaggle/working/ppd_outputs/cohort_prevalence_comparison.csv

=== Shared Feature Comparison (saved) ===
Saved: /kaggle/working/ppd_outputs/cohort_feature_comparison.csv
           feature  ante_mean  post_mean  ante_median  post_median  cohens_d_post_minus_ante  mannwhitney_p
  emotional_coping   5.037296   5.381199     5.000000     5.333333                  0.320805   1.352325e-17
anxious_attachment  41.933566  52.094077    38.000000    49.000000                  0.527258   2.851820e-15
   adaptive_coping   5.968531   5.853627     6.000000     6.000000                 -0.121758   3.872850e-04
maladaptive_coping   2.788811   2.716506     2.666667     2.666667                 -0.091575   2.282078e-01
 